In [ ]:
import pandas as pd
from google.colab import drive
import openpyxl

# 1. Mount Drive
drive.mount('/content/drive')




file_path = '/content/drive/MyDrive/traffic/crash/crashrate.csv'

# 3. Read the CSV file
countyweek = pd.read_csv(file_path)
print(countyweek.shape)
file_path = '/content/drive/MyDrive/traffic/crash/CrashStatistics.csv'

# 3. Read the CSV file
crashstatistics = pd.read_csv(file_path)


import pandas as pd
from google.colab import drive
import openpyxl

# 1. Mount Drive
drive.mount('/content/drive')




file_path = '/content/drive/MyDrive/traffic/alltime_combined_traffic_data.csv'

# 3. Read the CSV file
alltime = pd.read_csv(file_path)
#aggregate diff outcomes, compare crash rate per 100
#pull in covariates
# plot time series and distributions, log as well

# run diff and diff plus paralelle trend test

#run mixed effects

# cluster and county sensitivity analysis


data = countyweek.copy()

# Get Group and Storm_Week from alltime
group_lookup = alltime[
    ["County", "Group"]
].drop_duplicates().copy()

storm_lookup = alltime[
    ["County", "Week_Rel", "Storm_Week"]
].drop_duplicates().copy()



# Standardize County names
data["County"] = data["County"].str.strip().str.upper()
group_lookup["County"] = group_lookup["County"].str.strip().str.upper()
storm_lookup["County"] = storm_lookup["County"].str.strip().str.upper()

# Merge Group by County
data = data.drop(columns="Group", errors="ignore").merge(
    group_lookup,
    on="County",
    how="left",
    validate="many_to_one"
)
# Merge Storm_Week by County and Week_Rel
data = data.drop(columns="Storm_Week", errors="ignore").merge(
    storm_lookup,
    on=["County", "Week_Rel"],
    how="left",
    validate="many_to_one"
)
print(data.columns)



In [ ]:
# @title
import json
from urllib.request import urlopen

import pandas as pd
import plotly.express as px


# ---------------------------------------------------------
# 1. Clean your county-level campaign data
# ---------------------------------------------------------
campaign_df = alltime[['County', 'Campaign_Group']].drop_duplicates().reset_index(drop=True)

campaign_df["County"] = (
    campaign_df["County"]
    .astype(str)
    .str.strip()
    .str.replace(r"\s+County$", "", regex=True)
)


# ---------------------------------------------------------
# 2. Load US county boundaries
# ---------------------------------------------------------
geojson_url = (
    "https://raw.githubusercontent.com/plotly/datasets/"
    "master/geojson-counties-fips.json"
)

with urlopen(geojson_url) as response:
    us_counties = json.load(response)


# ---------------------------------------------------------
# 3. Keep only Ohio counties
# Ohio's state FIPS code is 39
# ---------------------------------------------------------
ohio_counties = {
    "type": "FeatureCollection",
    "features": [
        feature
        for feature in us_counties["features"]
        if feature["properties"]["STATE"] == "39"
    ],
}


# ---------------------------------------------------------
# 4. Create a County-to-FIPS lookup from the GeoJSON
# ---------------------------------------------------------
ohio_fips = pd.DataFrame(
    {
        "County": [
            feature["properties"]["NAME"]
            for feature in ohio_counties["features"]
        ],
        "fips": [
            feature["id"]
            for feature in ohio_counties["features"]
        ],
    }
)


# Keep every Ohio county, even if campaign data is missing
map_df = ohio_fips.merge(
    campaign_df,
    on="County",
    how="left",
    validate="one_to_one",
)

map_df["Campaign_Group"] = map_df["Campaign_Group"].fillna(
    "Missing / Unassigned"
)


# ---------------------------------------------------------
# 5. Check for county names that did not match
# ---------------------------------------------------------
unmatched = campaign_df.loc[
    ~campaign_df["County"].isin(ohio_fips["County"]),
    "County",
].unique()

if len(unmatched) > 0:
    print("County names that did not match:", unmatched)


# ---------------------------------------------------------
# 6. Plot the campaign groups
# ---------------------------------------------------------
color_map = {
    "Treated (Speeding)": "#E76F51",
    "Treated (Nighttime)": "#457B9D",
    "Control": "#BDBDBD",
    "Missing / Unassigned": "#F2F2F2",
}

fig = px.choropleth(
    map_df,
    geojson=ohio_counties,
    locations="fips",
    color="Campaign_Group",
    hover_name="County",
    hover_data={
        "fips": False,
        "Campaign_Group": True,
    },
    color_discrete_map=color_map,
    category_orders={
        "Campaign_Group": [
            "Treated (Speeding)",
            "Treated (Nighttime)",
            "Control",
            "Missing / Unassigned",
        ]
    },
    title="Ohio Counties by Campaign Group",
)

fig.update_geos(
    fitbounds="locations",
    visible=False,
)

fig.update_traces(
    marker_line_color="white",
    marker_line_width=0.8,
)

fig.update_layout(
    height=700,
    margin={"r": 0, "t": 60, "l": 0, "b": 0},
    legend_title_text="Campaign Group",
)

fig.show()

In [ ]:

import pandas as pd
import numpy as np
import plotly.express as px

# ---------------------------------------------------------
# 1. County-level treatment assignments
# ---------------------------------------------------------

county_meta = alltime[
    ["County", "Group", "Campaign_Group"]
].drop_duplicates().copy()

county_meta["County"] = (
    county_meta["County"]
    .astype(str)
    .str.strip()
    .str.replace(r"\s+COUNTY$", "", regex=True, case=False)
    .str.upper()
)

assert not county_meta["County"].duplicated().any(), (
    "Conflicting treatment assignments for some counties"
)

# Match county names to FIPS codes
fips_lookup = ohio_fips[["County", "fips"]].copy()
fips_lookup["County"] = fips_lookup["County"].str.strip().str.upper()
fips_lookup["fips"] = fips_lookup["fips"].astype(str).str.zfill(5)

county_meta = county_meta.merge(
    fips_lookup,
    on="County",
    how="left",
    validate="one_to_one"
)

assert county_meta["fips"].notna().all(), "Unmatched county FIPS codes"
assert len(county_meta) == 88, "Expected 88 Ohio counties"

# ---------------------------------------------------------
# 2. Census county adjacency data
# ---------------------------------------------------------

adjacency_url = (
    "https://www2.census.gov/geo/docs/reference/"
    "county_adjacency/county_adjacency2025.txt"
)

adj = pd.read_csv(
    adjacency_url,
    sep="|",
    dtype={
        "County GEOID": str,
        "Neighbor GEOID": str
    }
)

adj["Length"] = pd.to_numeric(adj["Length"], errors="coerce")

# Only Ohio-to-Ohio adjacency.
# Length > 0 excludes point-only contacts.
adj = adj[
    adj["County GEOID"].str.startswith("39")
    & adj["Neighbor GEOID"].str.startswith("39")
    & (adj["Length"] > 0)
].copy()

edges = adj[
    ["County GEOID", "Neighbor GEOID"]
].rename(columns={
    "County GEOID": "fips",
    "Neighbor GEOID": "neighbor_fips"
}).drop_duplicates()

# ---------------------------------------------------------
# 3. Attach assignments for both counties in each pair
# ---------------------------------------------------------

edges = edges.merge(
    county_meta[["fips", "Group"]],
    on="fips",
    how="inner",
    validate="many_to_one"
)

neighbor_lookup = county_meta[
    ["fips", "Group", "Campaign_Group"]
].rename(columns={
    "fips": "neighbor_fips",
    "Group": "Neighbor_Group",
    "Campaign_Group": "Neighbor_Campaign_Group"
})

edges = edges.merge(
    neighbor_lookup,
    on="neighbor_fips",
    how="inner",
    validate="many_to_one"
)

# CRITICAL: exclude neighbors in the same randomized cluster
edges = edges[
    edges["Group"] != edges["Neighbor_Group"]
].copy()

edges["treated_neighbor"] = (
    edges["Neighbor_Campaign_Group"] != "Control"
).astype(int)

edges["speed_neighbor"] = (
    edges["Neighbor_Campaign_Group"] == "Treated (Speeding)"
).astype(int)

edges["night_neighbor"] = (
    edges["Neighbor_Campaign_Group"] == "Treated (Nighttime)"
).astype(int)

# ---------------------------------------------------------
# 4. Aggregate neighbor exposure to the county level
# ---------------------------------------------------------

counts = edges.groupby("fips").agg(
    Cross_Cluster_Neighbors=("neighbor_fips", "nunique"),
    Treated_Neighbors=("treated_neighbor", "sum"),
    Speed_Neighbors=("speed_neighbor", "sum"),
    Night_Neighbors=("night_neighbor", "sum")
).reset_index()

exposure = county_meta.merge(
    counts,
    on="fips",
    how="left",
    validate="one_to_one"
)

count_cols = [
    "Cross_Cluster_Neighbors",
    "Treated_Neighbors",
    "Speed_Neighbors",
    "Night_Neighbors"
]

exposure[count_cols] = exposure[count_cols].fillna(0).astype(int)

# Binary exposure
exposure["Exposed"] = (
    exposure["Treated_Neighbors"] > 0
).astype(int)

# Continuous exposure: proportion of cross-cluster neighbors treated
exposure["Neighbor_Treated_Share"] = np.where(
    exposure["Cross_Cluster_Neighbors"] > 0,
    exposure["Treated_Neighbors"] /
    exposure["Cross_Cluster_Neighbors"].replace(0, np.nan),
    0.0
)

# ---------------------------------------------------------
# 5. Count exposed and unexposed control counties
# ---------------------------------------------------------

controls = exposure[
    exposure["Campaign_Group"] == "Control"
].copy()

print("Control-county exposure:")
print(controls["Exposed"].value_counts())

print("\nExposure by randomized control cluster:")

cluster_summary = controls.groupby("Group")["Exposed"].agg(
    Counties="count",
    Exposed="sum"
)

cluster_summary["Unexposed"] = (
    cluster_summary["Counties"] - cluster_summary["Exposed"]
)

display(cluster_summary)

print("\nGroup 13 exposure:")
display(
    controls.loc[
        controls["Group"] == 13,
        ["County", "Treated_Neighbors",
         "Neighbor_Treated_Share", "Exposed"]
    ]
)

exposure.to_csv("county_spillover_exposure.csv", index=False)

# ---------------------------------------------------------
# 6. Map exposed vs unexposed controls
# ---------------------------------------------------------

exposure["Map_Status"] = exposure["Campaign_Group"]

exposure.loc[
    (exposure["Campaign_Group"] == "Control")
    & (exposure["Exposed"] == 1),
    "Map_Status"
] = "Control: Exposed"

exposure.loc[
    (exposure["Campaign_Group"] == "Control")
    & (exposure["Exposed"] == 0),
    "Map_Status"
] = "Control: Unexposed"

fig = px.choropleth(
    exposure,
    geojson=ohio_counties,
    locations="fips",
    color="Map_Status",
    hover_name="County",
    hover_data=[
        "Group", "Treated_Neighbors",
        "Neighbor_Treated_Share"
    ],
    color_discrete_map={
        "Treated (Speeding)": "#E76F51",
        "Treated (Nighttime)": "#457B9D",
        "Control: Exposed": "#F4B860",
        "Control: Unexposed": "#BDBDBD"
    },
    title="Ohio County Spillover Exposure"
)

fig.update_geos(fitbounds="locations", visible=False)
fig.update_traces(marker_line_color="white", marker_line_width=0.7)
fig.update_layout(height=700)

fig.show()


In [ ]:

import matplotlib.pyplot as plt

# ---------------------------------------------------------
# 1. Prepare fatal crash dataset
# ---------------------------------------------------------

fatal_panel = data.copy()

fatal_panel["County"] = (
    fatal_panel["County"]
    .astype(str)
    .str.strip()
    .str.replace(r"\s+COUNTY$", "", regex=True, case=False)
    .str.upper()
)

fatal_panel = fatal_panel[
    fatal_panel["Severity"].astype(str).str.strip().str.upper()
    == "FATAL"
].copy()

# Attach geographic exposure variables
fatal_panel = fatal_panel.drop(
    columns=["Group", "Campaign_Group"],
    errors="ignore"
).merge(
    exposure[
        [
            "County", "Group", "Campaign_Group",
            "Exposed", "Neighbor_Treated_Share"
        ]
    ],
    on="County",
    how="left",
    validate="many_to_one"
)

assert fatal_panel["Group"].notna().all()
assert not fatal_panel.duplicated(
    ["County", "Week_Rel"]
).any(), "Duplicate county-week observations"

fatal_panel["Week_Rel"] = pd.to_numeric(
    fatal_panel["Week_Rel"]
).astype(int)

fatal_panel["Crashes"] = pd.to_numeric(fatal_panel["Crashes"])
fatal_panel["Population"] = pd.to_numeric(fatal_panel["Population"])

assert (fatal_panel["Population"] > 0).all()

# ---------------------------------------------------------
# 2. Restrict descriptive comparison to control counties
# ---------------------------------------------------------

control_panel = fatal_panel[
    fatal_panel["Campaign_Group"] == "Control"
].copy()

# Sum crashes and populations within each exposure group/week
trend = control_panel.groupby(
    ["Week_Rel", "Exposed"]
).agg(
    Crashes=("Crashes", "sum"),
    Population=("Population", "sum")
).reset_index()

trend["Fatal_Rate_per_100k"] = (
    trend["Crashes"] / trend["Population"] * 100000
)

# ---------------------------------------------------------
# 3. Plot observed trends
# ---------------------------------------------------------

fig, ax = plt.subplots(figsize=(11, 5))

for exposed, label in [
    (0, "Unexposed Controls"),
    (1, "Exposed Controls")
]:
    subset = trend[
        trend["Exposed"] == exposed
    ].sort_values("Week_Rel")

    if not subset.empty:
        ax.plot(
            subset["Week_Rel"],
            subset["Fatal_Rate_per_100k"],
            marker="o",
            linewidth=2,
            label=label
        )

# Weeks 0-1: campaign ramp-up
ax.axvspan(
    -0.5, 1.5,
    alpha=0.12,
    label="Ramp-up period"
)

ax.axvline(1.5, linestyle="--", color="gray")

ax.set_title(
    "Fatal Crash Rates: Exposed vs Unexposed Control Counties"
)
ax.set_xlabel("Week Relative to Campaign")
ax.set_ylabel("Weekly Fatal Crashes per 100,000 Residents")
ax.legend()
ax.grid(alpha=0.2)

plt.tight_layout()
plt.show()

display(trend)


In [ ]:

import statsmodels.api as sm
import statsmodels.formula.api as smf

# ---------------------------------------------------------
# 1. Define analysis windows
# ---------------------------------------------------------

PRE_WEEKS = [-4, -3, -2, -1, 0, 1]
POST_WEEKS = [2, 3, 4, 5]

model_data = fatal_panel[
    fatal_panel["Week_Rel"].isin(PRE_WEEKS + POST_WEEKS)
].copy()

model_data["Post_Spill"] = (
    model_data["Week_Rel"].isin(POST_WEEKS)
).astype(int)

model_data["Speeding"] = (
    model_data["Campaign_Group"] == "Treated (Speeding)"
).astype(int)

model_data["Nighttime"] = (
    model_data["Campaign_Group"] == "Treated (Nighttime)"
).astype(int)

# ---------------------------------------------------------
# 2. Reusable Poisson fixed-effects model
# ---------------------------------------------------------

def fit_spillover(df, exposure_variable, include_direct=False):

    if df[exposure_variable].nunique() < 2:
        print(f"Cannot estimate: no variation in {exposure_variable}")
        return None

    # Avoid a misleading binary-exposure estimate under extreme sparsity
    if exposure_variable == "Exposed":
        county_counts = (
            df[["County", "Exposed"]]
            .drop_duplicates()["Exposed"]
            .value_counts()
        )

        cell_crashes = df.groupby(
            ["Exposed", "Post_Spill"]
        )["Crashes"].sum()

        if county_counts.min() < 4 or (cell_crashes == 0).any():
            print("Binary comparison too sparse; try exposure share.")
            return None

    # County and week fixed effects
    rhs = (
        "C(County) + C(Week_Rel)"
        f" + {exposure_variable}:Post_Spill"
    )

    terms = [f"{exposure_variable}:Post_Spill"]

    if include_direct:
        rhs += " + Speeding:Post_Spill + Nighttime:Post_Spill"
        terms += [
            "Speeding:Post_Spill",
            "Nighttime:Post_Spill"
        ]

    model = smf.glm(
        formula="Crashes ~ " + rhs,
        data=df,
        family=sm.families.Poisson(),
        offset=np.log(df["Population"])
    )

    fit = model.fit(
        maxiter=150,
        cov_type="cluster",
        cov_kwds={"groups": df["Group"]}
    )

    ci = fit.conf_int()

    results = pd.DataFrame({
        "Coefficient": fit.params[terms],
        "Rate_Ratio": np.exp(fit.params[terms]),
        "Percent_Change": 100 * (np.exp(fit.params[terms]) - 1),
        "CI_Lower_pct": 100 * (np.exp(ci.loc[terms, 0]) - 1),
        "CI_Upper_pct": 100 * (np.exp(ci.loc[terms, 1]) - 1),
        "Nominal_P_Value": fit.pvalues[terms]
    })

    print(
        f"Counties: {df['County'].nunique()} | "
        f"Randomized clusters: {df['Group'].nunique()}"
    )

    display(results.round(4))

    return fit


# ---------------------------------------------------------
# Model A: Control-only binary spillover
# ---------------------------------------------------------

control_model = model_data[
    model_data["Campaign_Group"] == "Control"
].copy()

print("\nMODEL A: Binary geographic exposure")

fit_A = fit_spillover(
    control_model,
    "Exposed"
)


# ---------------------------------------------------------
# Model B: Continuous neighboring treatment exposure
# ---------------------------------------------------------

print("\nMODEL B: Proportion of treated neighbors")

fit_B = fit_spillover(
    control_model,
    "Neighbor_Treated_Share"
)


# ---------------------------------------------------------
# Model C: Remove influential Group 13
# ---------------------------------------------------------

print("\nMODEL C: Excluding Group 13")

fit_C = fit_spillover(
    control_model[control_model["Group"] != 13],
    "Neighbor_Treated_Share"
)


# ---------------------------------------------------------
# Model D: Full-sample model including direct treatment
# ---------------------------------------------------------

print("\nMODEL D: Direct treatment + geographic spillover")

fit_D = fit_spillover(
    model_data,
    "Neighbor_Treated_Share",
    include_direct=True
)


# ---------------------------------------------------------
# Model E: Exclude two ramp-up weeks
# ---------------------------------------------------------

print("\nMODEL E: Excluding ramp-up weeks")

fit_E = fit_spillover(
    control_model[
        ~control_model["Week_Rel"].isin([0, 1])
    ],
    "Neighbor_Treated_Share"
)

# Save analysis panel
model_data.to_csv(
    "county_fatal_spillover_panel.csv",
    index=False
)

print(
    "\nCaution: Only five randomized control clusters. "
    "Cluster-robust confidence intervals and nominal p-values "
    "are exploratory, not definitive causal spillover inference."
)

#In our analysis, a county is defined as exposed to potential spillover if it shares a geographic
# boundary with at least one county assigned to a treated cluster (speeding or nighttime campaign).

ran this model onjust control cities:

crash count ~ post/pre + binary(border or not) + beta_j* post/pre:binary +

#a little work and its insignificant


In [ ]:
#conducted a geographic spillover sensitivity analysis using control counties.
#We found no statistically significant association between neighboring treatment exposure and fatal crash rates, either with or without Group 13.
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# =========================================================
# 1. GROUP 13: EXPOSED VS UNEXPOSED FATAL CRASH RATES
# =========================================================

g13 = control_model[
    control_model["Group"] == 13
].copy()

g13["Period"] = np.where(
    g13["Post_Spill"] == 1, "Post", "Pre"
)

summary = (
    g13.groupby(["Exposed", "Period"])
    .agg(
        Counties=("County", "nunique"),
        Fatal_Crashes=("Crashes", "sum"),
        Population_Weeks=("Population", "sum")
    )
    .reset_index()
)

summary["Rate_per_100k"] = (
    100000 * summary["Fatal_Crashes"]
    / summary["Population_Weeks"]
)

summary["Exposure_Group"] = summary["Exposed"].map({
    0: "Unexposed Controls",
    1: "Exposed Controls"
})

# Pivot for comparison
rates = summary.pivot(
    index="Exposure_Group",
    columns="Period",
    values="Rate_per_100k"
).reindex(["Unexposed Controls", "Exposed Controls"])

rates["Percent_Change"] = (
    100 * (rates["Post"] / rates["Pre"] - 1)
)

print("GROUP 13: PRE/POST COMPARISON")
display(rates.round(4))


# =========================================================
# 2. PLOT GROUP 13 PRE/POST COMPARISON
# =========================================================

fig, ax = plt.subplots(figsize=(9, 5))

x = np.arange(len(rates))
width = 0.35

bars_pre = ax.bar(
    x - width/2,
    rates["Pre"],
    width,
    label="Pre"
)

bars_post = ax.bar(
    x + width/2,
    rates["Post"],
    width,
    label="Post"
)

ax.bar_label(bars_pre, fmt="%.3f", padding=3)
ax.bar_label(bars_post, fmt="%.3f", padding=3)

ax.set_xticks(x)
ax.set_xticklabels(rates.index)

ax.set_ylabel("Weekly Fatal Crashes per 100,000 Residents")
ax.set_title(
    "Group 13: Fatal Crash Rates by Geographic Spillover Exposure"
)

ax.legend()
ax.grid(axis="y", alpha=0.2)
ax.set_axisbelow(True)

plt.tight_layout()
plt.show()


# =========================================================
# 3. SENSITIVITY: WITH VS WITHOUT GROUP 13
# =========================================================

term = "Neighbor_Treated_Share:Post_Spill"

def extract_estimate(fit, label):
    beta = fit.params[term]
    ci = fit.conf_int().loc[term]

    return {
        "Model": label,
        "Coefficient": beta,
        "Percent_Change": 100 * (np.exp(beta) - 1),
        "CI_Lower_pct": 100 * (np.exp(ci.iloc[0]) - 1),
        "CI_Upper_pct": 100 * (np.exp(ci.iloc[1]) - 1),
        "P_Value": fit.pvalues[term]
    }

sensitivity = pd.DataFrame([
    extract_estimate(fit_B, "All Control Counties"),
    extract_estimate(fit_C, "Excluding Group 13")
])

print("\nSPILLOVER SENSITIVITY ANALYSIS")
display(sensitivity.round(4))


# =========================================================
# 4. OPTIONAL: SAVE OUTPUTS
# =========================================================

rates.to_csv("group13_prepost_rates.csv")
sensitivity.to_csv("group13_spillover_sensitivity.csv", index=False)


In [ ]:

import pandas as pd
import numpy as np
import patsy
import matplotlib.pyplot as plt
import statsmodels.api as sm

from scipy.stats import poisson
from scipy.special import expit
from statsmodels.discrete.count_model import ZeroInflatedPoisson

# =========================================================
# 1. PREPARE FATAL CRASH DATA
# =========================================================

assignment = alltime[
    ["County", "Group", "Campaign_Group"]
].drop_duplicates().copy()

assignment["County"] = (
    assignment["County"].astype(str).str.strip().str.upper()
    .str.replace(r"\s+COUNTY$", "", regex=True)
)

assert not assignment["County"].duplicated().any()

fatal = data[
    data["Severity"].astype(str).str.strip().str.upper() == "FATAL"
].copy()

fatal["County"] = (
    fatal["County"].astype(str).str.strip().str.upper()
    .str.replace(r"\s+COUNTY$", "", regex=True)
)

fatal = fatal.drop(
    columns=["Group", "Campaign_Group"], errors="ignore"
).merge(
    assignment,
    on="County",
    how="left",
    validate="many_to_one"
)

assert fatal["Group"].notna().all()
assert fatal["Campaign_Group"].notna().all()

PRE_WEEKS = [-4, -3, -2, -1, 0, 1]
POST_WEEKS = [2, 3, 4, 5]

fatal["Week_Rel"] = pd.to_numeric(
    fatal["Week_Rel"]
).astype(int)

fatal = fatal[
    fatal["Week_Rel"].isin(PRE_WEEKS + POST_WEEKS)
].copy()

fatal["Crashes"] = pd.to_numeric(fatal["Crashes"])
fatal["Population"] = pd.to_numeric(fatal["Population"])

assert (fatal["Population"] > 0).all()
assert not fatal.duplicated(["County", "Week_Rel"]).any()

fatal["Post"] = fatal["Week_Rel"].isin(POST_WEEKS).astype(int)

fatal["Speeding"] = (
    fatal["Campaign_Group"] == "Treated (Speeding)"
).astype(int)

fatal["Nighttime"] = (
    fatal["Campaign_Group"] == "Treated (Nighttime)"
).astype(int)

assert fatal["Campaign_Group"].isin([
    "Control", "Treated (Speeding)", "Treated (Nighttime)"
]).all()

# Population offset, scaled per 100,000 residents
offset = np.log(fatal["Population"] / 100000)

# =========================================================
# 2. COMMON COUNT-MODEL SPECIFICATION
# =========================================================

formula = (
    "1 + C(Group)"
    " + C(Week_Rel)"
    " + Speeding:Post"
    " + Nighttime:Post"
)

X = patsy.dmatrix(formula, fatal, return_type="dataframe")
y = fatal["Crashes"].astype(int)

print("Observations:", len(fatal))
print("Counties:", fatal["County"].nunique())
print("Randomized clusters:", fatal["Group"].nunique())
print("Observed zero proportion:", round((y == 0).mean(), 4))

# =========================================================
# 3. STANDARD POISSON MODEL
# =========================================================

poisson_fit = sm.GLM(
    endog=y,
    exog=X,
    family=sm.families.Poisson(),
    offset=offset
).fit()

# =========================================================
# 4. ZERO-INFLATED POISSON MODEL
# =========================================================

# Count part: identical covariates to standard Poisson.
# Inflation part: intercept-only logistic model.
#
# ZIP mean = (1 - pi) * lambda

zip_model = ZeroInflatedPoisson(
    endog=y,
    exog=X,
    exog_infl=np.ones((len(y), 1)),
    inflation="logit",
    offset=offset
)

zip_fit = zip_model.fit(
    method="bfgs",
    maxiter=500,
    disp=False
)

print("\nZIP converged:",
      zip_fit.mle_retvals.get("converged", False))

# =========================================================
# 5. COMPARE CAMPAIGN EFFECTS
# =========================================================

terms = ["Speeding:Post", "Nighttime:Post"]

results = pd.DataFrame({
    "Campaign": ["Speeding", "Nighttime"],
    "Poisson_Coefficient": poisson_fit.params.loc[terms].values,
    "ZIP_Coefficient": zip_fit.params.loc[terms].values
})

results["Poisson_Percent_Change"] = (
    100 * np.expm1(results["Poisson_Coefficient"])
)

results["ZIP_Percent_Change"] = (
    100 * np.expm1(results["ZIP_Coefficient"])
)

print("\nCAMPAIGN EFFECT ESTIMATES")
display(results.round(4))

# =========================================================
# 6. COMPARE FIT AND ZERO PREDICTIONS
# =========================================================

poisson_mean = np.asarray(poisson_fit.predict())

zip_zero_prob = np.asarray(
    zip_fit.predict(which="prob-zero")
)

zip_inflation_prob = float(
    expit(zip_fit.params["inflate_const"])
)

comparison = pd.DataFrame({
    "Model": ["Poisson", "Zero-Inflated Poisson"],
    "AIC": [poisson_fit.aic, zip_fit.aic],
    "Predicted_Zero_Proportion": [
        np.exp(-poisson_mean).mean(),
        zip_zero_prob.mean()
    ]
})

print("\nMODEL COMPARISON")
display(comparison.round(4))

print("Observed zero proportion:", round((y == 0).mean(), 4))
print("Estimated extra-zero probability:",
      round(zip_inflation_prob, 6))

# =========================================================
# 7. OBSERVED VS PREDICTED CRASH DISTRIBUTION
# =========================================================

# ZIP Poisson-component mean
zip_lambda = np.asarray(
    zip_fit.predict(which="mean-main")
)

observed = np.array([
    (y == 0).sum(),
    (y == 1).sum(),
    (y == 2).sum(),
    (y >= 3).sum()
])

pred_poisson = np.array([
    poisson.pmf(k, poisson_mean).sum()
    for k in [0, 1, 2]
])

pred_poisson = np.append(
    pred_poisson, len(y) - pred_poisson.sum()
)

pred_zip = np.array([
    zip_zero_prob.sum(),
    (1 - zip_inflation_prob) *
        poisson.pmf(1, zip_lambda).sum(),
    (1 - zip_inflation_prob) *
        poisson.pmf(2, zip_lambda).sum()
])

pred_zip = np.append(
    pred_zip, len(y) - pred_zip.sum()
)

distribution = pd.DataFrame({
    "Crash_Count": ["0", "1", "2", "3+"],
    "Observed": observed,
    "Poisson": pred_poisson,
    "ZIP": pred_zip
})

display(distribution.round(2))

x = np.arange(4)
width = 0.25

fig, ax = plt.subplots(figsize=(9, 5))

ax.bar(x - width, observed, width, label="Observed")
ax.bar(x, pred_poisson, width, label="Poisson")
ax.bar(x + width, pred_zip, width, label="ZIP")

ax.set_xticks(x)
ax.set_xticklabels(["0", "1", "2", "3+"])
ax.set_xlabel("Fatal Crashes per County-Week")
ax.set_ylabel("Number of County-Week Observations")
ax.set_title("Observed vs Predicted Fatal Crash Distribution")
ax.legend()
ax.grid(axis="y", alpha=0.2)

plt.tight_layout()
plt.show()


In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Group 13 only
g13 = control_model[control_model["Group"] == 13].copy()

# Aggregate pre/post crashes and population
county_summary = (
    g13.groupby(["County", "Exposed", "Post_Spill"])
    .agg(
        Crashes=("Crashes", "sum"),
        Population_Weeks=("Population", "sum")
    )
    .reset_index()
)

county_summary["Rate"] = (
    100000 * county_summary["Crashes"]
    / county_summary["Population_Weeks"]
)

# Reshape to one row per county
county_comparison = county_summary.pivot(
    index=["County", "Exposed"],
    columns="Post_Spill",
    values=["Crashes", "Rate"]
)

county_comparison.columns = [
    f"{metric}_{'Post' if period == 1 else 'Pre'}"
    for metric, period in county_comparison.columns
]

county_comparison = county_comparison.reset_index()

# Calculate change in fatal crash rate
county_comparison["Rate_Change"] = (
    county_comparison["Rate_Post"]
    - county_comparison["Rate_Pre"]
)

county_comparison["Exposure_Group"] = np.where(
    county_comparison["Exposed"] == 1,
    "Exposed",
    "Unexposed"
)

display(
    county_comparison[
        ["County", "Exposure_Group",
         "Crashes_Pre", "Crashes_Post",
         "Rate_Pre", "Rate_Post", "Rate_Change"]
    ].sort_values("Rate_Change").round(4)
)

# Plot county-level rate changes
plot_df = county_comparison.sort_values("Rate_Change")

fig, ax = plt.subplots(figsize=(10, 7))

colors = plot_df["Exposed"].map({
    1: "steelblue",
    0: "gray"
})

ax.barh(
    plot_df["County"],
    plot_df["Rate_Change"],
    color=colors
)

ax.axvline(0, color="black", linewidth=1)

ax.set_xlabel("Post - Pre Fatal Crash Rate per 100,000")
ax.set_title("Group 13: County-Level Changes in Fatal Crash Rates")
ax.grid(axis="x", alpha=0.2)

plt.tight_layout()
plt.show()


In [ ]:
# @title Weather and Demographic Covariates

import pandas as pd
import numpy as np
import meteostat as ms
from datacommons_client.client import DataCommonsClient

# Assumes `client` is already initialized with your API key.


client = DataCommonsClient(
    api_key=__import__("os").environ["DATACOMMONS_API_KEY"]
)
# ---------------------------------------------------------
# Get Ohio counties from Data Commons
# ---------------------------------------------------------

response = client.node.fetch_property_values(
    node_dcids=["geoId/39"],
    properties="containedInPlace+{typeOf:County}",
    out=False
).to_dict()

county_nodes = (
    response["data"]["geoId/39"]["arcs"]
    ["containedInPlace+"]["nodes"]
)

county_locations = pd.DataFrame(
    county_nodes
)[["dcid", "name"]]

county_locations["County"] = (
    county_locations["name"]
    .str.replace(" County", "", regex=False)
    .str.strip()
    .str.upper()
)


# ---------------------------------------------------------
# Get county coordinates
# ---------------------------------------------------------

coordinate_response = (
    client.node.fetch_property_values(
        node_dcids=county_locations["dcid"].tolist(),
        properties=["latitude", "longitude"]
    )
    .to_dict()["data"]
)


def get_coordinate(dcid, variable):

    nodes = (
        coordinate_response[dcid]["arcs"]
        [variable]["nodes"]
    )

    return float(nodes[0]["value"])


county_locations["Latitude"] = (
    county_locations["dcid"]
    .apply(lambda x: get_coordinate(x, "latitude"))
)

county_locations["Longitude"] = (
    county_locations["dcid"]
    .apply(lambda x: get_coordinate(x, "longitude"))
)


# ---------------------------------------------------------
# Clean countyweek dates and counties
# ---------------------------------------------------------

countyweek["Week_Start"] = pd.to_datetime(
    countyweek["Week_Start"],
    format="mixed"
).dt.normalize()

countyweek["County"] = (
    countyweek["County"]
    .astype(str)
    .str.replace(
        r"\s+County$",
        "",
        regex=True,
        case=False
    )
    .str.strip()
    .str.upper()
)

start_date = countyweek["Week_Start"].min().date()

end_date = (
    countyweek["Week_Start"].max()
    + pd.Timedelta(days=6)
).date()


# ---------------------------------------------------------
# Download weather data
# ---------------------------------------------------------

weather_data = []
missing_counties = []

for _, county in county_locations.iterrows():

    point = ms.Point(
        float(county["Latitude"]),
        float(county["Longitude"])
    )

    stations = ms.stations.nearby(
        point,
        radius=150_000,
        limit=4
    )

    if stations is None or len(stations) == 0:

        missing_counties.append(county["County"])

        print(f"No stations found for {county['County']}")

        continue

    daily = ms.daily(
        stations,
        start_date,
        end_date,
        parameters=[
            ms.Parameter.TEMP,
            ms.Parameter.TMIN,
            ms.Parameter.TMAX,
            ms.Parameter.PRCP,
            ms.Parameter.SNWD,
            ms.Parameter.WSPD,
            ms.Parameter.WPGT
        ]
    )

    daily = ms.interpolate(
        daily,
        point
    ).fetch()

    if daily.empty:

        print(f"No weather data found for {county['County']}")
        continue

    daily.index.name = "Date"
    daily = daily.reset_index()

    daily.columns = [
        getattr(column, "value", column)
        for column in daily.columns
    ]

    daily["County"] = county["County"]

    weather_data.append(daily)


print("Counties without stations:", missing_counties)

if len(weather_data) == 0:
    raise ValueError("No weather data were downloaded.")


# ---------------------------------------------------------
# Combine daily weather
# ---------------------------------------------------------

weather_daily = pd.concat(
    weather_data,
    ignore_index=True
)

weather_daily["Date"] = pd.to_datetime(
    weather_daily["Date"]
)


# ---------------------------------------------------------
# Assign Tuesday-start weeks
# ---------------------------------------------------------

weather_daily["Week_Start"] = (
    weather_daily["Date"].dt.normalize()
    - pd.to_timedelta(
        (weather_daily["Date"].dt.dayofweek - 1) % 7,
        unit="D"
    )
)


# ---------------------------------------------------------
# Aggregate daily weather to weekly weather
# ---------------------------------------------------------

weather_weekly = (
    weather_daily
    .groupby(
        ["County", "Week_Start"],
        as_index=False
    )
    .agg(
        temp_c=("temp", "mean"),
        precipitation_mm=("prcp", "sum"),
        snow_depth_cm=("snwd", "max"),
        wind_speed_kph=("wspd", "mean")
    )
)


# ---------------------------------------------------------
# Get 2024 demographic data from Data Commons
# ---------------------------------------------------------

demographic_variables = {
    "Median_Income_Household":
        "median_household_income",

    "UnemploymentRate_Person":
        "unemployment_rate",

    "Count_Person":
        "population",

    "Count_Person_18To24Years":
        "population_18_24",

    "Count_Person_65OrMoreYears":
        "population_65_plus",

    "Count_Person_25OrMoreYears":
        "population_25_plus",

    "Count_Person_25OrMoreYears_EducationalAttainmentBachelorsDegreeOrHigher":
        "population_bachelors_plus"
}


demographics_long = client.observations_dataframe(
    variable_dcids=list(demographic_variables.keys()),
    date="2024",
    entity_dcids=county_locations["dcid"].tolist(),
    property_filters={
        "measurementMethod": "CensusACS5yrSurvey"
    }
)


demographics = (
    demographics_long
    .pivot_table(
        index="entity",
        columns="variable",
        values="value",
        aggfunc="first"
    )
    .reindex(columns=demographic_variables.keys())
    .rename(columns=demographic_variables)
    .reset_index()
    .rename(columns={"entity": "dcid"})
)


demographics = county_locations[
    ["dcid", "County"]
].merge(
    demographics,
    on="dcid",
    how="left",
    validate="one_to_one"
)


# ---------------------------------------------------------
# Convert demographic data to numeric
# ---------------------------------------------------------

numeric_demo_cols = list(demographic_variables.values())

demographics[numeric_demo_cols] = (
    demographics[numeric_demo_cols]
    .apply(pd.to_numeric, errors="coerce")
)


# ---------------------------------------------------------
# Create model-ready demographic variables
# ---------------------------------------------------------

demographics["log_median_household_income"] = np.log(
    demographics["median_household_income"].where(
        demographics["median_household_income"] > 0
    )
)

demographics["pct_age_18_24"] = (
    100
    * demographics["population_18_24"]
    / demographics["population"]
)

demographics["pct_age_65_plus"] = (
    100
    * demographics["population_65_plus"]
    / demographics["population"]
)

demographics["pct_bachelors_plus"] = (
    100
    * demographics["population_bachelors_plus"]
    / demographics["population_25_plus"]
)


demographic_columns = [
    "County",
    "log_median_household_income",
    "unemployment_rate",
    "pct_age_18_24",
    "pct_age_65_plus",
    "pct_bachelors_plus"
]


# ---------------------------------------------------------
# Merge weather and demographics into countyweek
# ---------------------------------------------------------

weather_cols = [
    "temp_c",
    "precipitation_mm",
    "snow_depth_cm",
    "wind_speed_kph"
]

# Make sure Week_Start has the same data type
data["Week_Start"] = pd.to_datetime(
    data["Week_Start"], format="mixed"
).dt.normalize()

weather_weekly["Week_Start"] = pd.to_datetime(
    weather_weekly["Week_Start"], format="mixed"
).dt.normalize()

# Avoid duplicate columns if rerunning
data = data.drop(
    columns=weather_cols + demographic_columns[1:],
    errors="ignore"
)

data = data.merge(
    weather_weekly[
        ["County", "Week_Start"] + weather_cols
    ],
    on=["County", "Week_Start"],
    how="left",
    validate="many_to_one"
)

data = data.merge(
    demographics[demographic_columns],
    on="County",
    how="left",
    validate="many_to_one"
)


# ---------------------------------------------------------
# Validate merges
# ---------------------------------------------------------

print("Original rows:", len(countyweek))
print("Merged rows:", len(data))

print("\nMissing weather values:")
display(data[weather_cols].isna().sum())

print("\nMissing demographic values:")
display(data[demographic_columns[1:]].isna().sum())

display(
    data[
        [
            "County",
            "Week_Start",
            "temp_c",
            "precipitation_mm",
            "snow_depth_cm",
            "wind_speed_kph",
            "log_median_household_income",
            "unemployment_rate",
            "pct_age_18_24",
            "pct_age_65_plus",
            "pct_bachelors_plus"
        ]
    ].head(15)
)

In [ ]:
! pip install meteostat

In [ ]:
data['log_Population']= np.log1p(data['Population'])
data['log_pct_age_18_24']= np.log1p(data['pct_age_18_24'])
data['log_pct_age_65_plus']= np.log1p(data['pct_age_65_plus'])
data['log_precipitation_mm']= np.log1p(data['precipitation_mm'])
data['log_wind_speed_kph']= np.log1p(data['wind_speed_kph'])
data['log_Crash_Rate_per_100k']= np.log1p(data['Crash_Rate_per_100k'])
data["temp_f"] = data["temp_c"] * 9/5 + 32
data["log_temp_f"] = np.log1p(data["temp_f"])

In [ ]:
data.to_csv()
import pandas as pd
from google.colab import drive
import openpyxl

# 1. Mount Drive
drive.mount('/content/drive')




file_path = '/content/drive/MyDrive/traffic/crash/crashrate.csv'
data.to_csv('/content/drive/MyDrive/traffic/crash/data.csv', index=False)

In [ ]:
import statsmodels.formula.api as smf
from scipy.stats import chi2
from statsmodels.stats.multitest import multipletests

# Refer to this: https://stackoverflow.com/questions/50052421/mixed-models-with-two-random-effects-statsmodels
metric = 'Crash_Rate_per_100k'
for sev in data['Severity'].unique():
    nighttime_wide = data[data['Severity'] == sev]
    nighttime_wide["Post"] = (nighttime_wide["Week_Rel"] >= 2).astype(int)
    formula = metric + ' ~ C(Arm) + C(Post) + C(Arm):C(Post)'
    formula0 = metric + ' ~ C(Arm) + C(Post) '

    vcf = {'re1': '0 + C(Group)'}
    nighttime_wide['group'] = 1 # All in the same group
    print(formula, "formula")
    mm = smf.mixedlm(formula, nighttime_wide.dropna(subset=[metric]), groups = nighttime_wide.dropna(subset=[metric])['group'],
                    vc_formula=vcf, re_formula="0")
    mm0 = smf.mixedlm(formula0, nighttime_wide.dropna(subset=[metric]), groups = nighttime_wide.dropna(subset=[metric])['group'],
                     vc_formula=vcf, re_formula="0")

    mm_fit = mm.fit(reml=False)
    mm_fit0 = mm0.fit(reml=False)

    p_val = 1 - chi2.cdf(-2*(mm_fit0.llf-mm_fit.llf), df=1)
    lr_stat = 2 * (
        mm_fit.llf - mm_fit0.llf
    )

    p_val = chi2.sf(
        lr_stat,
        df=2
    )
    print(mm_fit.params, "mm_fit.params")
    trt_1_fitted_lift = np.exp(mm_fit.params['C(Arm)[T.Nighttime]:C(Post)[T.1]']) - 1
    trt_1_fitted_lift_se = np.exp(mm_fit.bse['C(Arm)[T.Nighttime]:C(Post)[T.1]']) - 1
    trt_1_fitted_lift_p = mm_fit.pvalues['C(Arm)[T.Nighttime]:C(Post)[T.1]']
    trt_2_fitted_lift = np.exp(mm_fit.params['C(Arm)[T.Speeding]:C(Post)[T.1]']) - 1
    trt_2_fitted_lift_se = np.exp(mm_fit.bse['C(Arm)[T.Speeding]:C(Post)[T.1]']) - 1
    trt_2_fitted_lift_p = mm_fit.pvalues['C(Arm)[T.Speeding]:C(Post)[T.1]']
    print('In {},  {} to {}, treatment 1 saw a {}% (+/- {}%) impact relative to control. p-value: {} ' \
    .format(metric, 'pre_start_date', 'post_end_date', round(trt_1_fitted_lift * 100, 2), round(trt_1_fitted_lift_se * 100, 2), round(trt_1_fitted_lift_p, 4)))
    print('In {},  {} to {}, treatment 2 saw a {}% (+/- {}%) impact relative to control. p-value: {} ' \
    .format(metric, 'pre_start_date', 'post_end_date', round(trt_2_fitted_lift * 100, 2), round(trt_2_fitted_lift_se * 100, 2), round(trt_2_fitted_lift_p, 4)))
    raw_pvalues = [
        trt_1_fitted_lift_p,
        trt_2_fitted_lift_p
    ]

    bonferroni_pvalues = multipletests(
        raw_pvalues,
        method="bonferroni"
    )[1]

    holm_pvalues = multipletests(
        raw_pvalues,
        method="holm"
    )[1]
    print(
    f"In {metric}, treatment 1 saw a "
    f"{trt_1_fitted_lift * 100:.2f}% "
    f"(+/- {trt_1_fitted_lift_se * 100:.2f}%) impact. "
    f"Raw p-value: {trt_1_fitted_lift_p:.4f}, "
    f"Bonferroni p-value: {bonferroni_pvalues[0]:.4f}, "
    f"Holm p-value: {holm_pvalues[0]:.4f}"
    )

    print(
        f"In {metric}, treatment 2 saw a "
        f"{trt_2_fitted_lift * 100:.2f}% "
        f"(+/- {trt_2_fitted_lift_se * 100:.2f}%) impact. "
        f"Raw p-value: {trt_2_fitted_lift_p:.4f}, "
        f"Bonferroni p-value: {bonferroni_pvalues[1]:.4f}, "
        f"Holm p-value: {holm_pvalues[1]:.4f}"
    )
    diagnostic_df = plot_mixed_diagnostics(
      mm_fit,
      nighttime_wide.dropna(subset=[metric]),
      metric
    )


In [ ]:
# @title OLS DiD vs Mixed Effects DiD by Crash Severity

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.formula.api as smf

from scipy import stats
from scipy.stats import chi2
from statsmodels.stats.multitest import multipletests

metric = 'log_Crash_Rate_per_100k'

results = []
fitted_models = {}

for sev in data['Severity'].unique():

    nighttime_wide = data[data['Severity'] == sev].copy()
    nighttime_wide["Post"] = (nighttime_wide["Week_Rel"] >= 2).astype(int)

    formula = metric + " ~ C(Arm) + C(Post) +log_Population +log_temp_f+ log_pct_age_18_24 +  log_pct_age_65_plus+ log_precipitation_mm+ log_wind_speed_kph+ C(Arm):C(Post)"
    formula0 = metric + " ~ C(Arm) + C(Post) +log_Population + log_temp_f+log_pct_age_18_24+ log_pct_age_65_plus+ log_precipitation_mm + log_wind_speed_kph"

    nighttime_wide["Arm"] = pd.Categorical(
        nighttime_wide["Arm"],
        categories=["Control", "Nighttime", "Speeding"]
    )

    nighttime_wide = nighttime_wide.dropna(
        subset=[metric, "Arm", "Post", "Group"]
    ).copy()

    vcf = {'re1': '0 + C(Group)'}
    nighttime_wide['group'] = 1

    print(f"\n{'=' * 60}")
    print(sev, "Severity")
    print(formula, "formula")

    # ---------------------------------------------------------
    # Mixed-effects model (original specification)
    # ---------------------------------------------------------

    mm = smf.mixedlm(
        formula,
        nighttime_wide,
        groups=nighttime_wide['group'],
        vc_formula=vcf,
        re_formula="0"
    )

    mm0 = smf.mixedlm(
        formula0,
        nighttime_wide,
        groups=nighttime_wide['group'],
        vc_formula=vcf,
        re_formula="0"
    )

    mm_fit = mm.fit(reml=False)
    mm_fit0 = mm0.fit(reml=False)

    lr_stat = 2 * (mm_fit.llf - mm_fit0.llf)
    p_val = chi2.sf(max(0, lr_stat), df=2)

    # ---------------------------------------------------------
    # OLS DiD — same fixed-effects formula
    # ---------------------------------------------------------

    ols_fit = smf.ols(
        formula,
        nighttime_wide
    ).fit(
        cov_type="cluster",
        cov_kwds={
            "groups": nighttime_wide["Group"],
            "use_correction": True,
            "df_correction": True
        },
        use_t=True
    )

    fitted_models[sev] = {
        "Mixed": mm_fit,
        "OLS": ols_fit
    }

    # ---------------------------------------------------------
    # Treatment effects for both models
    # ---------------------------------------------------------

    for model_name, fit in [
        ("Mixed", mm_fit),
        ("OLS", ols_fit)
    ]:

        for treatment in ["Nighttime", "Speeding"]:

            term = (
                f"C(Arm)[T.{treatment}]"
                ":C(Post)[T.1]"
            )

            ci = fit.conf_int().loc[term]

            results.append({
                "Severity": sev,
                "Model": model_name,
                "Treatment": treatment,
                "Estimate": fit.params[term],
                "SE": fit.bse[term],
                "CI_Lower": ci.iloc[0],
                "CI_Upper": ci.iloc[1],
                "P_Value": fit.pvalues[term],
                "AIC": fit.aic,
                "BIC": fit.bic,
                "Log_Likelihood": fit.llf,
                "Joint_LR_P": p_val if model_name == "Mixed" else np.nan
            })

    # ---------------------------------------------------------
    # Mixed-effects diagnostics — your existing function
    # ---------------------------------------------------------

    diagnostic_df = plot_mixed_diagnostics(
        mm_fit,
        nighttime_wide,
        metric
    )

    # ---------------------------------------------------------
    # OLS diagnostic plots
    # ---------------------------------------------------------

    residuals = np.asarray(ols_fit.resid)
    fitted = np.asarray(ols_fit.fittedvalues)

    fig, axs = plt.subplots(2, 2, figsize=(12, 8))

    sns.scatterplot(
        x=fitted, y=residuals, ax=axs[0, 0]
    )
    axs[0, 0].axhline(
        0, color="black", linestyle="--"
    )
    axs[0, 0].set_title("OLS: Residuals vs Fitted")

    stats.probplot(
        residuals, dist="norm", plot=axs[0, 1]
    )
    axs[0, 1].set_title("OLS: Q-Q Plot")

    sns.histplot(
        residuals, kde=True, ax=axs[1, 0]
    )
    axs[1, 0].set_title("OLS: Residual Distribution")

    sns.scatterplot(
        x=fitted,
        y=np.abs(residuals),
        ax=axs[1, 1]
    )
    axs[1, 1].set_title("OLS: Absolute Residuals vs Fitted")

    fig.suptitle(
        f"{sev} — OLS DiD Diagnostics",
        fontsize=15
    )

    plt.tight_layout()
    plt.show()
    plt.close(fig)


# ---------------------------------------------------------
# Comparison table
# ---------------------------------------------------------

results_df = pd.DataFrame(results)

results_df["Bonferroni_P"] = np.nan
results_df["Holm_P"] = np.nan

for _, idx in results_df.groupby(
    ["Severity", "Model"]
).groups.items():

    raw_pvalues = results_df.loc[idx, "P_Value"]

    results_df.loc[idx, "Bonferroni_P"] = multipletests(
        raw_pvalues,
        method="bonferroni"
    )[1]

    results_df.loc[idx, "Holm_P"] = multipletests(
        raw_pvalues,
        method="holm"
    )[1]


pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)

display(
    results_df.sort_values(
        ["Severity", "Treatment", "Model"]
    ).round(4)
)

In [ ]:
# @title Poisson DiD — Fatal Crashes

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
import statsmodels.formula.api as smf

from statsmodels.stats.multitest import multipletests

# ---------------------------------------------------------
# Setup
# ---------------------------------------------------------

fatal_data = data[
    (data["Severity"] == "Fatal")
    & (data["Week_Rel"].between(-4, 5))
].copy()

fatal_data["Post"] = (
    fatal_data["Week_Rel"] >= 2
).astype(int)

fatal_data["Arm"] = pd.Categorical(
    fatal_data["Arm"],
    categories=["Control", "Nighttime", "Speeding"]
)


# ---------------------------------------------------------
# Fit Poisson DiD
# ---------------------------------------------------------

formula = (
    "Crashes ~ C(Arm)"
    " + C(Post)+ log_temp_f+ log_pct_age_18_24 +  log_pct_age_65_plus+ log_precipitation_mm+ log_wind_speed_kph"
    " + C(Arm):C(Post)"
)



poisson_fit = smf.glm(
    formula=formula,
    data=fatal_data,
    family=sm.families.Poisson(),
    offset=np.log(fatal_data["Population"])
).fit(
    cov_type="cluster",
    cov_kwds={
        "groups": fatal_data["Group"]
    }
)

# Arm-specific post-treatment indicators
fatal_data["Nighttime_Post"] = (
    (fatal_data["Arm"] == "Nighttime") &
    (fatal_data["Post"] == 1)
).astype(int)

fatal_data["Speeding_Post"] = (
    (fatal_data["Arm"] == "Speeding") &
    (fatal_data["Post"] == 1)
).astype(int)

formula_week_fe = (
    "Crashes ~ C(Arm) + C(Week_Rel)"
    " + log_temp_f + log_pct_age_18_24"
    " + log_pct_age_65_plus"
    " + log_precipitation_mm"
    " + log_wind_speed_kph"
    " + Nighttime_Post + Speeding_Post"
)
poisson_fit = smf.glm(
    formula=formula_week_fe,
    data=fatal_data,
    family=sm.families.Poisson(),
    offset=np.log(fatal_data["Population"])
).fit(
    cov_type="cluster",
    cov_kwds={"groups": fatal_data["Group"]}
)

print(poisson_fit.summary())
from scipy import stats

# Observed and predicted fatal crash counts
y = np.asarray(poisson_fit.model.endog, dtype=float)
mu = np.asarray(poisson_fit.fittedvalues, dtype=float)

# Pearson residuals
pearson_resid = (y - mu) / np.sqrt(mu)

# Pearson chi-square statistic
pearson_chi2 = np.sum(pearson_resid**2)

# Residual degrees of freedom (n - number of parameters)
df_resid = int(poisson_fit.df_resid)

# Dispersion ratio
dispersion_ratio = pearson_chi2 / df_resid

# Upper-tail chi-square p-value
p_value = stats.chi2.sf(pearson_chi2, df_resid)

print(f"Pearson Chi-square: {pearson_chi2:.3f}")
print(f"Residual DF: {df_resid}")
print(f"Dispersion Ratio: {dispersion_ratio:.3f}")
print(f"P-value: {p_value:.6g}")

print(poisson_fit.summary())

# ---------------------------------------------------------
# Extract treatment effects
# ---------------------------------------------------------

results = []

for treatment in ["Nighttime", "Speeding"]:

    term = (
        f"C(Arm)[T.{treatment}]"
        ":C(Post)[T.1]"
    )

    beta = poisson_fit.params[term]
    ci = poisson_fit.conf_int().loc[term]

    results.append({
        "Treatment": treatment,
        "Coefficient": beta,
        "IRR": np.exp(beta),
        "Percent_Change": 100 * np.expm1(beta),
        "CI_Lower_Percent": 100 * np.expm1(ci.iloc[0]),
        "CI_Upper_Percent": 100 * np.expm1(ci.iloc[1]),
        "P_Value": poisson_fit.pvalues[term]
    })

poisson_results = pd.DataFrame(results)

poisson_results["Holm_P"] = multipletests(
    poisson_results["P_Value"],
    method="holm"
)[1]

print("\nPoisson DiD Treatment Effects")
display(poisson_results.round(4))

# ---------------------------------------------------------
# Diagnostic plots
# ---------------------------------------------------------

fitted = np.asarray(poisson_fit.fittedvalues)
residuals = np.asarray(poisson_fit.resid_pearson)

fig, axs = plt.subplots(
    nrows=1,
    ncols=2,
    figsize=(13, 5)
)

sns.scatterplot(
    x=fitted,
    y=residuals,
    ax=axs[0]
)

axs[0].axhline(
    0,
    color="black",
    linestyle="--"
)

axs[0].set_title("Pearson Residuals vs Fitted Counts")
axs[0].set_xlabel("Fitted Fatal Crash Counts")
axs[0].set_ylabel("Pearson Residuals")

sns.histplot(
    residuals,
    kde=True,
    ax=axs[1]
)

axs[1].set_title("Pearson Residual Distribution")
axs[1].set_xlabel("Pearson Residual")

plt.tight_layout()
plt.show()
plt.close(fig)

# ---------------------------------------------------------
# Overdispersion and zero-count diagnostics
# ---------------------------------------------------------

dispersion = (
    poisson_fit.pearson_chi2
    / poisson_fit.df_resid
)

observed_zero_pct = (
    (fatal_data["Crashes"] == 0).mean() * 100
)

expected_zero_pct = (
    np.exp(-fitted).mean() * 100
)

print("\nModel Diagnostics")
print("AIC:", round(poisson_fit.aic, 3))
print("BIC (log-likelihood based):", round(poisson_fit.bic_llf, 3))
print("Log-likelihood:", round(poisson_fit.llf, 3))
print("Dispersion statistic:", round(dispersion, 3))
print("Observed zero-count percentage:", round(observed_zero_pct, 2))
print("Expected zero-count percentage:", round(expected_zero_pct, 2))

In [ ]:
# Residuals by treatment arm and week
used_rows = poisson_fit.model.data.row_labels

diagnostic_data = fatal_data.loc[
    used_rows, ["Arm", "Week_Rel"]
].copy()

diagnostic_data["Pearson_Residual"] = pearson_resid

weekly_arm_resid = (
    diagnostic_data
    .groupby(["Arm", "Week_Rel"], observed=True)["Pearson_Residual"]
    .mean()
    .reset_index()
)

plt.figure(figsize=(10, 5))

sns.lineplot(
    data=weekly_arm_resid,
    x="Week_Rel",
    y="Pearson_Residual",
    hue="Arm",
    marker="o"
)

plt.axhline(0, color="red", linestyle="--")
plt.xlabel("Week Relative to Intervention")
plt.ylabel("Mean Pearson Residual")
plt.title("Poisson DiD Residuals by Arm and Week")
plt.show()

In [ ]:

# Residuals and fitted values
deviance_resid = np.asarray(poisson_fit.resid_deviance)
pearson_resid = np.asarray(poisson_fit.resid_pearson)
log_fitted = np.log(mu)

# Influence diagnostics
influence = poisson_fit.get_influence(observed=True)
leverage = np.asarray(influence.hat_matrix_diag)
std_pearson = np.asarray(influence.resid_studentized)

fig, axes = plt.subplots(2, 2, figsize=(12, 9))

# 1. Deviance residuals vs fitted
axes[0, 0].scatter(log_fitted, deviance_resid, alpha=0.5)
axes[0, 0].axhline(0, color="red", linestyle="--")
axes[0, 0].set_title("Residuals vs Fitted")
axes[0, 0].set_xlabel("Log fitted count")
axes[0, 0].set_ylabel("Deviance Residuals")

# 2. Normal Q-Q plot
stats.probplot(deviance_resid, dist="norm", plot=axes[0, 1])
axes[0, 1].set_title("Normal Q-Q Plot")

# 3. Scale-location
axes[1, 0].scatter(
    log_fitted,
    np.sqrt(np.abs(deviance_resid)),
    alpha=0.5
)
axes[1, 0].set_title("Scale-Location")
axes[1, 0].set_xlabel("Log fitted count")
axes[1, 0].set_ylabel("Sqrt(|Deviance Residuals|)")

# 4. Residuals vs leverage
axes[1, 1].scatter(leverage, std_pearson, alpha=0.5)
axes[1, 1].axhline(0, color="red", linestyle="--")
axes[1, 1].set_title("Residuals vs Leverage")
axes[1, 1].set_xlabel("Leverage")
axes[1, 1].set_ylabel("Standardized Pearson Residuals")

plt.tight_layout()
plt.show()


def plot_binned_residuals(x, residuals, ax, nbins=12):
    order = np.argsort(x)
    groups = np.array_split(order, min(nbins, len(x)))

    x_mean = [np.mean(x[g]) for g in groups if len(g)]
    r_mean = [np.mean(residuals[g]) for g in groups if len(g)]

    ax.plot(x_mean, r_mean, marker="o")
    ax.axhline(0, color="red", linestyle="--")


fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Binned Pearson residuals vs fitted
plot_binned_residuals(
    log_fitted, pearson_resid, axes[0]
)
axes[0].set_title("Binned Residuals vs Fitted")
axes[0].set_xlabel("Log fitted count")
axes[0].set_ylabel("Mean Pearson Residual")

# Average residual by relative week
used_rows = poisson_fit.model.data.row_labels
weeks = fatal_data.loc[used_rows, "Week_Rel"].to_numpy()

week_resid = pd.DataFrame({
    "Week_Rel": weeks,
    "Residual": pearson_resid
})

week_avg = (
    week_resid.groupby("Week_Rel")["Residual"]
    .mean()
)

axes[1].plot(
    week_avg.index,
    week_avg.values,
    marker="o"
)
axes[1].axhline(0, color="red", linestyle="--")
axes[1].set_title("Mean Residual by Week")
axes[1].set_xlabel("Week Relative to Intervention")
axes[1].set_ylabel("Mean Pearson Residual")

plt.tight_layout()
plt.show()



In [ ]:

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Observed vs predicted
axes[0].scatter(mu, y, alpha=0.5)
axes[0].set_title("Observed vs Predicted Fatal Crashes")
axes[0].set_xlabel("Expected Crashes")
axes[0].set_ylabel("Observed Crashes")

# Log-transformed version from lecture
axes[1].scatter(
    np.log1p(mu),
    np.log1p(y),
    alpha=0.5
)
axes[1].set_title("Log Observed vs Log Predicted")
axes[1].set_xlabel("log(Expected Crashes + 1)")
axes[1].set_ylabel("log(Observed Crashes + 1)")

plt.tight_layout()
plt.show()


In [ ]:

observed_zeros = np.sum(y == 0)
expected_zeros = np.sum(np.exp(-mu))

print(f"Observed zeros: {observed_zeros}")
print(f"Expected zeros: {expected_zeros:.2f}")
print(f"Observed zero proportion: {np.mean(y == 0):.3f}")
print(f"Expected zero proportion: {np.mean(np.exp(-mu)):.3f}")
print(f"Observed / Expected zeros: {observed_zeros / expected_zeros:.3f}")

# Simulation-based model checks
rng = np.random.default_rng(123)
n_sims = 2000

sim_zeros = np.empty(n_sims)
sim_dispersion = np.empty(n_sims)

for i in range(n_sims):
    # Simulate counts under fitted Poisson model
    y_sim = rng.poisson(mu)

    # Number of zeros
    sim_zeros[i] = np.sum(y_sim == 0)

    # Pearson dispersion relative to fixed fitted means
    sim_dispersion[i] = np.sum(
        (y_sim - mu)**2 / mu
    ) / len(y)

# Observed statistics
obs_zeros = np.sum(y == 0)
obs_dispersion = pearson_chi2 / len(y)

# Upper-tail Monte Carlo p-values
p_zero = (
    1 + np.sum(sim_zeros >= obs_zeros)
) / (n_sims + 1)

p_dispersion = (
    1 + np.sum(sim_dispersion >= obs_dispersion)
) / (n_sims + 1)

print(f"Zero-inflation check p-value: {p_zero:.4f}")
print(f"Dispersion simulation p-value: {p_dispersion:.4f}")

# Plot simulated null distributions
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].hist(sim_zeros, bins=30, edgecolor="black")
axes[0].axvline(
    obs_zeros, color="red", linewidth=2,
    label="Observed"
)
axes[0].set_title("Zero-Inflation Check")
axes[0].set_xlabel("Number of Zero Counts")
axes[0].set_ylabel("Frequency")
axes[0].legend()

axes[1].hist(sim_dispersion, bins=30, edgecolor="black")
axes[1].axvline(
    obs_dispersion, color="red", linewidth=2,
    label="Observed"
)
axes[1].set_title("Poisson Dispersion Check")
axes[1].set_xlabel("Pearson Dispersion (X² / n)")
axes[1].set_ylabel("Frequency")
axes[1].legend()

plt.tight_layout()
plt.show()



In [ ]:
# @title Poisson DiD — Group and Time Window Sensitivity

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
import statsmodels.formula.api as smf

from statsmodels.stats.multitest import multipletests

# ---------------------------------------------------------
# Setup — same Poisson specification
# ---------------------------------------------------------

formula = (
    "Crashes ~ C(Arm)"
    " + C(Post)"
    " + log_temp_f"
    " + log_pct_age_18_24"
    " + log_pct_age_65_plus"
    " + log_precipitation_mm"
    " + log_wind_speed_kph"
    " + C(Arm):C(Post)"
)

fatal_data = data[
    data["Severity"] == "Fatal"
].copy()

fatal_data["Arm"] = pd.Categorical(
    fatal_data["Arm"],
    categories=["Control", "Nighttime", "Speeding"]
)

required = [
    "Crashes", "Population", "Arm", "Group",
    "Week_Rel", "log_temp_f",
    "log_pct_age_18_24", "log_pct_age_65_plus",
    "log_precipitation_mm", "log_wind_speed_kph"
]

fatal_data = fatal_data.dropna(subset=required).copy()
fatal_data = fatal_data[fatal_data["Population"] > 0].copy()

# ---------------------------------------------------------
# Function to fit Poisson DiD
# ---------------------------------------------------------

def fit_poisson_did(model_df):

    model_df = model_df.copy()

    if model_df["Arm"].nunique() < 3:
        return []

    fit = smf.glm(
        formula=formula,
        data=model_df,
        family=sm.families.Poisson(),
        offset=np.log(model_df["Population"])
    ).fit(
        cov_type="cluster",
        cov_kwds={
            "groups": model_df["Group"]
        }
    )

    output = []

    for treatment in ["Nighttime", "Speeding"]:

        term = (
            f"C(Arm)[T.{treatment}]"
            ":C(Post)[T.1]"
        )

        beta = fit.params[term]
        ci = fit.conf_int().loc[term]

        output.append({
            "Treatment": treatment,
            "Coefficient": beta,
            "IRR": np.exp(beta),
            "Percent_Change": 100 * np.expm1(beta),
            "CI_Lower": 100 * np.expm1(ci.iloc[0]),
            "CI_Upper": 100 * np.expm1(ci.iloc[1]),
            "P_Value": fit.pvalues[term],
            "AIC": fit.aic,
            "BIC": fit.bic_llf,
            "Log_Likelihood": fit.llf,
            "N": int(fit.nobs)
        })

    adjusted = multipletests(
        [r["P_Value"] for r in output],
        method="holm"
    )[1]

    for row, p in zip(output, adjusted):
        row["Holm_P"] = p

    return output


# ---------------------------------------------------------
# 1. Leave-one-Group-out sensitivity
# ---------------------------------------------------------

original = fatal_data[
    fatal_data["Week_Rel"].between(-4, 5)
].copy()

original["Post"] = (
    original["Week_Rel"] >= 2
).astype(int)

group_results = []

# Full sample
for result in fit_poisson_did(original):
    group_results.append({
        "Omitted_Group": "Full Sample",
        **result
    })

# Remove one randomized Group at a time
for group in sorted(original["Group"].unique()):

    subset = original[
        original["Group"] != group
    ].copy()

    try:
        for result in fit_poisson_did(subset):

            group_results.append({
                "Omitted_Group": str(group),
                **result
            })

    except Exception as e:
        print(f"Group {group} failed: {e}")

group_results_df = pd.DataFrame(group_results)

print("\nLEAVE-ONE-GROUP-OUT RESULTS")

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)

display(group_results_df.round(4))


# ---------------------------------------------------------
# 2. Time-window sensitivity
# ---------------------------------------------------------

periods = {
    "Original": (
        [-4, -3, -2, -1, 0, 1],
        [2, 3, 4, 5]
    ),
    "No Ramp-Up": (
        [-4, -3, -2, -1],
        [2, 3, 4, 5]
    ),
    "Exclude First Week": (
        [-3, -2, -1, 0, 1],
        [2, 3, 4, 5]
    ),
    "Extended Post": (
        [-4, -3, -2, -1, 0, 1],
        [2, 3, 4, 5, 6]
    )
}

period_results = []

for period_name, (pre_weeks, post_weeks) in periods.items():

    subset = fatal_data[
        fatal_data["Week_Rel"].isin(
            pre_weeks + post_weeks
        )
    ].copy()

    subset["Post"] = (
        subset["Week_Rel"].isin(post_weeks)
    ).astype(int)

    try:
        for result in fit_poisson_did(subset):

            period_results.append({
                "Period": period_name,
                **result
            })

    except Exception as e:
        print(f"{period_name} failed: {e}")

period_results_df = pd.DataFrame(period_results)

print("\nTIME-WINDOW SENSITIVITY RESULTS")
display(period_results_df.round(4))


# ---------------------------------------------------------
# 3. Leave-one-Group-out plots
# ---------------------------------------------------------

sns.set_theme(style="whitegrid")

for treatment in ["Nighttime", "Speeding"]:

    subset = group_results_df[
        group_results_df["Treatment"] == treatment
    ].copy()

    full_effect = subset.loc[
        subset["Omitted_Group"] == "Full Sample",
        "Percent_Change"
    ].iloc[0]

    plot_df = subset[
        subset["Omitted_Group"] != "Full Sample"
    ].sort_values("Percent_Change")

    fig, ax = plt.subplots(figsize=(10, 6))

    ax.scatter(
        plot_df["Percent_Change"],
        plot_df["Omitted_Group"]
    )

    ax.axvline(
        full_effect,
        color="red",
        linestyle="--",
        label="Full sample"
    )

    ax.axvline(
        0,
        color="black",
        linestyle=":"
    )

    ax.set_title(
        f"{treatment} — Leave-One-Group-Out"
    )
    ax.set_xlabel("Estimated Fatal Crash Rate Change (%)")
    ax.set_ylabel("Omitted Group")
    ax.legend()

    plt.tight_layout()
    plt.show()
    plt.close(fig)


# ---------------------------------------------------------
# 4. Time-window comparison plots
# ---------------------------------------------------------

for treatment in ["Nighttime", "Speeding"]:

    subset = period_results_df[
        period_results_df["Treatment"] == treatment
    ].copy()

    fig, ax = plt.subplots(figsize=(10, 5))

    for i, (_, row) in enumerate(subset.iterrows()):

        ax.errorbar(
            row["Percent_Change"],
            i,
            xerr=[
                [row["Percent_Change"] - row["CI_Lower"]],
                [row["CI_Upper"] - row["Percent_Change"]]
            ],
            fmt="o",
            capsize=4
        )

    ax.set_yticks(range(len(subset)))
    ax.set_yticklabels(subset["Period"])

    ax.axvline(
        0,
        color="black",
        linestyle="--"
    )

    ax.set_title(
        f"{treatment} — Time Window Sensitivity"
    )

    ax.set_xlabel(
        "Estimated Fatal Crash Rate Change (%)"
    )

    plt.tight_layout()
    plt.show()
    plt.close(fig)


# ---------------------------------------------------------
# 5. Identify influential Groups
# ---------------------------------------------------------

full_effects = (
    group_results_df[
        group_results_df["Omitted_Group"] == "Full Sample"
    ][["Treatment", "Percent_Change"]]
    .rename(columns={
        "Percent_Change": "Full_Effect"
    })
)

influence_df = group_results_df[
    group_results_df["Omitted_Group"] != "Full Sample"
].merge(
    full_effects,
    on="Treatment",
    how="left"
)

influence_df["Change"] = (
    influence_df["Percent_Change"]
    - influence_df["Full_Effect"]
)

influence_df["Absolute_Change"] = (
    influence_df["Change"].abs()
)

print("\nMOST INFLUENTIAL GROUPS")

display(
    influence_df.sort_values(
        "Absolute_Change",
        ascending=False
    ).groupby("Treatment").head(5).round(4)
)

In [ ]:
# @title Investigate Group 13 — Fatal Crash Rates

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# ---------------------------------------------------------
# Setup
# ---------------------------------------------------------

df = original.copy()

df["Group"] = pd.to_numeric(df["Group"])
df["Post"] = (df["Week_Rel"] >= 2).astype(int)

# ---------------------------------------------------------
# 1. Group 13 information
# ---------------------------------------------------------

group13 = df[df["Group"] == 13].copy()

print("Group 13 treatment arm:")
print(group13["Arm"].unique())

print("\nCounties in Group 13:")
print(group13["County"].unique())

print("\nNumber of counties:", group13["County"].nunique())
print("Population:", group13.groupby("County")["Population"].first().sum())


# ---------------------------------------------------------
# 2. Aggregate fatal crashes by Group and Week
# ---------------------------------------------------------

group_week = (
    df.groupby(
        ["Group", "Arm", "Week_Rel", "Post"],
        observed=True,
        as_index=False
    )
    .agg(
        Crashes=("Crashes", "sum"),
        Population=("Population", "sum")
    )
)

group_week["Fatal_Rate"] = (
    group_week["Crashes"]
    / group_week["Population"]
    * 100_000
)


# ---------------------------------------------------------
# 3. Compare pre/post changes across Control Groups
# ---------------------------------------------------------

control = group_week[
    group_week["Arm"] == "Control"
].copy()

control_summary = (
    control.groupby(["Group", "Post"])
    .agg(
        Average_Weekly_Rate=("Fatal_Rate", "mean"),
        Total_Fatal_Crashes=("Crashes", "sum")
    )
    .reset_index()
)

control_rates = (
    control_summary
    .pivot(
        index="Group",
        columns="Post",
        values="Average_Weekly_Rate"
    )
    .rename(columns={0: "Pre", 1: "Post"})
)

control_rates["Change"] = (
    control_rates["Post"] - control_rates["Pre"]
)

print("\nCONTROL GROUP PRE/POST COMPARISON")
display(control_rates.sort_values("Change"))


# ---------------------------------------------------------
# 4. Group 13 vs all other Control Groups
# ---------------------------------------------------------

df["Region"] = np.where(
    df["Group"] == 13,
    "Group 13",
    np.where(
        df["Arm"] == "Control",
        "Other Controls",
        df["Arm"].astype(str)
    )
)

region_week = (
    df.groupby(
        ["Region", "Week_Rel", "Post"],
        as_index=False
    )
    .agg(
        Crashes=("Crashes", "sum"),
        Population=("Population", "sum")
    )
)

region_week["Fatal_Rate"] = (
    region_week["Crashes"]
    / region_week["Population"]
    * 100_000
)

region_summary = (
    region_week.groupby(["Region", "Post"])
    ["Fatal_Rate"]
    .mean()
    .unstack()
    .rename(columns={0: "Pre", 1: "Post"})
)

region_summary["Change"] = (
    region_summary["Post"] - region_summary["Pre"]
)

print("\nGROUP 13 VS OTHER REGIONS")
display(region_summary)


# ---------------------------------------------------------
# 5. Weekly time series
# ---------------------------------------------------------

fig, axs = plt.subplots(1, 2, figsize=(16, 5))

# Group 13 versus other regions
sns.lineplot(
    data=region_week,
    x="Week_Rel",
    y="Fatal_Rate",
    hue="Region",
    marker="o",
    ax=axs[0]
)

axs[0].axvline(1.5, color="black", linestyle="--")
axs[0].set_title("Group 13 vs Other Regions")
axs[0].set_ylabel("Fatal Crashes per 100,000")
axs[0].set_xlabel("Week_Rel")

# Individual Control Groups
for group, group_data in control.groupby("Group"):

    axs[1].plot(
        group_data["Week_Rel"],
        group_data["Fatal_Rate"],
        marker="o",
        label=f"Group {group}",
        linewidth=2.5 if group == 13 else 1.5
    )

axs[1].axvline(1.5, color="black", linestyle="--")
axs[1].set_title("Individual Control Group Trajectories")
axs[1].set_ylabel("Fatal Crashes per 100,000")
axs[1].set_xlabel("Week_Rel")
axs[1].legend()

plt.tight_layout()
plt.show()


# ---------------------------------------------------------
# 6. Identify influential counties within Group 13
# ---------------------------------------------------------

county_summary = (
    group13.groupby(["County", "Post"])
    .agg(
        Crashes=("Crashes", "sum"),
        Population=("Population", "first"),
        Weeks=("Week_Rel", "nunique")
    )
    .reset_index()
)

county_summary["Average_Weekly_Rate"] = (
    county_summary["Crashes"]
    / county_summary["Weeks"]
    / county_summary["Population"]
    * 100_000
)

county_rates = (
    county_summary
    .pivot(
        index="County",
        columns="Post",
        values="Average_Weekly_Rate"
    )
    .rename(columns={0: "Pre", 1: "Post"})
)

county_rates["Change"] = (
    county_rates["Post"] - county_rates["Pre"]
)

print("\nGROUP 13 — COUNTY PRE/POST CHANGES")
display(county_rates.sort_values("Change"))

# ---------------------------------------------------------
# 7. Plot county-level changes
# ---------------------------------------------------------

plt.figure(figsize=(10, 7))

county_rates["Change"].sort_values().plot(
    kind="barh"
)

plt.axvline(0, color="black", linestyle="--")

plt.title("Group 13: Change in Weekly Fatal Crash Rate")
plt.xlabel("Post - Pre (per 100,000)")
plt.ylabel("County")

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

for severity in data["Severity"].unique():

    df_sev = data[data["Severity"] == severity]

    print(f"\n{severity} Crash Distribution")
    print(df_sev["Crashes"].describe())

    print("Percent zero crashes:",
          round((df_sev["Crashes"] == 0).mean() * 100, 2))

    fig, axs = plt.subplots(1, 2, figsize=(14, 5))

    sns.histplot(
        df_sev["Crashes"],
        bins=30,
        kde=True,
        ax=axs[0]
    )
    axs[0].set_title(f"{severity}: Crash Count Distribution")

    sns.boxplot(
        x=df_sev["Crashes"],
        ax=axs[1]
    )
    axs[1].set_title(f"{severity}: Boxplot")

    plt.tight_layout()
    plt.show()
    plt.close(fig)

In [ ]:
# @title
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats


def plot_mixed_diagnostics(mm_fit, model_df, metric):

    # Rows actually used by Statsmodels
    used_rows = mm_fit.model.data.row_labels
    diagnostic_df = model_df.loc[used_rows].copy().reset_index(drop=True)

    y = np.asarray(mm_fit.model.endog)

    # Marginal fitted values: fixed effects only, X beta
    marginal_fitted = (
        mm_fit.model.exog @ np.asarray(mm_fit.fe_params)
    )

    marginal_residuals = y - marginal_fitted

    # Conditional fitted values: fixed + predicted random effects
    conditional_fitted = np.asarray(mm_fit.fittedvalues)
    conditional_residuals = np.asarray(mm_fit.resid)

    standardized_residuals = (
        conditional_residuals / np.sqrt(mm_fit.scale)
    )

    diagnostic_df["Marginal"] = marginal_residuals
    diagnostic_df["Conditional"] = conditional_residuals

    fig, axs = plt.subplots(
        nrows=2,
        ncols=3,
        figsize=(18, 10)
    )

    # 1. Marginal residuals versus fixed-effects fitted values
    sns.regplot(
        x=marginal_fitted,
        y=marginal_residuals,
        lowess=True,
        scatter_kws={"alpha": 0.25},
        line_kws={"color": "red"},
        ax=axs[0, 0]
    )

    axs[0, 0].axhline(0, color="black", linestyle="--")
    axs[0, 0].set_title("Marginal residuals")
    axs[0, 0].set_xlabel("Fixed-effects fitted values")
    axs[0, 0].set_ylabel("Marginal residual")

    # 2. Conditional residuals versus full fitted values
    sns.regplot(
        x=conditional_fitted,
        y=conditional_residuals,
        lowess=True,
        scatter_kws={"alpha": 0.25},
        line_kws={"color": "red"},
        ax=axs[0, 1]
    )

    axs[0, 1].axhline(0, color="black", linestyle="--")
    axs[0, 1].set_title("Conditional residuals")
    axs[0, 1].set_xlabel("Conditional fitted values")
    axs[0, 1].set_ylabel("Conditional residual")

    # 3. Normal Q-Q plot for conditional residuals
    stats.probplot(
        conditional_residuals,
        dist="norm",
        plot=axs[0, 2]
    )

    axs[0, 2].set_title("Q-Q plot: conditional residuals")

    # 4. Marginal and conditional residuals by Group
    group_residuals = diagnostic_df[
        ["Group", "Marginal", "Conditional"]
    ].melt(
        id_vars="Group",
        var_name="Residual type",
        value_name="Residual"
    )

    sns.boxplot(
        data=group_residuals,
        x="Group",
        y="Residual",
        hue="Residual type",
        showfliers=False,
        ax=axs[1, 0]
    )

    axs[1, 0].axhline(0, color="black", linestyle="--")
    axs[1, 0].set_title("Residuals by randomized Group")
    axs[1, 0].tick_params(axis="x", rotation=45)

    # 5. Standardized conditional residuals
    axs[1, 1].scatter(
        np.arange(len(standardized_residuals)),
        standardized_residuals,
        alpha=0.3
    )

    axs[1, 1].axhline(0, color="black", linestyle="--")
    axs[1, 1].axhline(3, color="red", linestyle="--")
    axs[1, 1].axhline(-3, color="red", linestyle="--")
    axs[1, 1].set_title("Standardized conditional residuals")
    axs[1, 1].set_xlabel("Observation")
    axs[1, 1].set_ylabel("Standardized residual")

    # 6. Normal Q-Q plot for Group random effects
    try:
        random_effect_values = []

        for random_effect in mm_fit.random_effects.values():

            names = pd.Index(
                random_effect.index
            ).astype(str)

            # Extract the re1 = Group variance component
            keep = names.str.startswith("re1")

            random_effect_values.extend(
                np.asarray(random_effect)[keep]
            )

        stats.probplot(
            random_effect_values,
            dist="norm",
            plot=axs[1, 2]
        )

        axs[1, 2].set_title("Q-Q plot: Group random effects")

    except Exception as error:
        axs[1, 2].axis("off")
        axs[1, 2].text(
            0.5,
            0.5,
            f"Random effects unavailable:\n{error}",
            ha="center",
            va="center"
        )

    fig.suptitle(
        f"Mixed-model diagnostics: {metric}",
        fontsize=15
    )

    plt.tight_layout()
    plt.show()

    return diagnostic_df

In [ ]:
# @title OLS vs Mixed Effects DiD with Diagnostic Plots by Severity

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.formula.api as smf
import warnings

from scipy import stats
from scipy.stats import chi2
from statsmodels.stats.multitest import multipletests

# ---------------------------------------------------------
# Setup
# ---------------------------------------------------------
data = countyweek_cov.copy()

# Get Group and Storm_Week from alltime
group_lookup = alltime[
    ["County", "Group"]
].drop_duplicates().copy()

storm_lookup = alltime[
    ["County", "Week_Rel", "Storm_Week"]
].drop_duplicates().copy()

# Standardize County names
data["County"] = data["County"].str.strip().str.upper()
group_lookup["County"] = group_lookup["County"].str.strip().str.upper()
storm_lookup["County"] = storm_lookup["County"].str.strip().str.upper()

# Merge Group by County
data = data.drop(columns="Group", errors="ignore").merge(
    group_lookup,
    on="County",
    how="left",
    validate="many_to_one"
)

# Merge Storm_Week by County and Week_Rel
data = data.drop(columns="Storm_Week", errors="ignore").merge(
    storm_lookup,
    on=["County", "Week_Rel"],
    how="left",
    validate="many_to_one"
)

data["Arm"] = pd.Categorical(
    data["Arm"],
    categories=["Control", "Nighttime", "Speeding"]
)

# Reference: weeks -4 to 1
# Analysis: weeks 2 to 5
data = data[data["Week_Rel"].between(-4, 5)].copy()

data["Post"] = (data["Week_Rel"] >= 2).astype(int)

print(data['Group'].value_counts())

metric = "log_Crash_Rate_per_100k"

formula = (
    metric
    + " ~ C(Arm)"
    + " + C(Post)"
    + " + log_Population"
    + " + C(Storm_Week)"
    + " + temp_c"
    + " + log_pct_age_18_24"
    + " + log_pct_age_65_plus"
    + " + log_precipitation_mm"
    + " + log_wind_speed_kph"
    + " + C(Arm):C(Post)"
)

formula0 = (
    metric
    + " ~ C(Arm)"
    + " + C(Post)"
    + " + log_Population"
    + " + C(Storm_Week)"
    + " + temp_c"
    + " + log_pct_age_18_24"
    + " + log_pct_age_65_plus"
    + " + log_precipitation_mm"
    + " + log_wind_speed_kph"
)

required = [
    metric, "Arm", "Post", "Group",
    "Storm_Week", "log_Population",
    "temp_c", "log_pct_age_18_24",
    "log_pct_age_65_plus",
    "log_precipitation_mm", "log_wind_speed_kph"
]

results = []
fitted_models = {}

# ---------------------------------------------------------
# Fit models for each Severity
# ---------------------------------------------------------

for severity in data["Severity"].dropna().unique():

    print(f"\n{'=' * 60}")
    print(f"Severity: {severity}")
    print(f"{'=' * 60}")

    model_df = data[
        data["Severity"] == severity
    ].dropna(subset=required).copy()

    # OLS DiD - Group-clustered standard errors
    ols_fit = smf.ols(
        formula,
        data=model_df
    ).fit(
        cov_type="cluster",
        cov_kwds={
            "groups": model_df["Group"],
            "use_correction": True,
            "df_correction": True
        },
        use_t=True
    )

    # Mixed Effects DiD - Group random intercept
    mm_fit = None
    mm_fit0 = None
    joint_p = np.nan

    try:
        mm_fit = smf.mixedlm(
            formula,
            model_df,
            groups=model_df["Group"],
            re_formula="1"
        ).fit(
            reml=False,
            method="lbfgs",
            maxiter=2000
        )

        mm_fit0 = smf.mixedlm(
            formula0,
            model_df,
            groups=model_df["Group"],
            re_formula="1"
        ).fit(
            reml=False,
            method="lbfgs",
            maxiter=2000
        )

        if mm_fit.converged and mm_fit0.converged:
            lr_stat = 2 * (mm_fit.llf - mm_fit0.llf)
            joint_p = chi2.sf(max(0, lr_stat), df=2)

    except (ValueError, np.linalg.LinAlgError) as e:
        print(f"Mixed model issue: {e}")

    fitted_models[severity] = {
        "OLS": ols_fit,
        "Mixed": mm_fit,
        "data": model_df,
        "Joint_LR_P": joint_p
    }

    # -----------------------------------------------------
    # Extract treatment effects
    # -----------------------------------------------------

    for model_name, fit in [
        ("OLS", ols_fit),
        ("Mixed", mm_fit)
    ]:

        if fit is None:
            continue

        for treatment in ["Nighttime", "Speeding"]:

            term = (
                f"C(Arm)[T.{treatment}]"
                ":C(Post)[T.1]"
            )

            ci = fit.conf_int().loc[term]

            results.append({
                "Severity": severity,
                "Model": model_name,
                "Treatment": treatment,
                "Estimate": fit.params[term],
                "SE": fit.bse[term],
                "CI_Lower": ci.iloc[0],
                "CI_Upper": ci.iloc[1],
                "P_Value": fit.pvalues[term],
                "Joint_LR_P": joint_p if model_name == "Mixed" else np.nan,
                "Converged": (
                    fit.converged if model_name == "Mixed" else True
                )
            })

# ---------------------------------------------------------
# Results table
# ---------------------------------------------------------

results_df = pd.DataFrame(results)
results_df["Holm_P"] = np.nan

for _, idx in results_df.groupby(
    ["Severity", "Model"]
).groups.items():

    pvals = results_df.loc[idx, "P_Value"]

    if pvals.notna().all():
        results_df.loc[idx, "Holm_P"] = multipletests(
            pvals,
            method="holm"
        )[1]

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)

display(results_df.round(4))


# ---------------------------------------------------------
# Diagnostic plots: OLS vs Mixed, side by side
# ---------------------------------------------------------

def get_diagnostics(fit, model_name):

    if model_name == "Mixed":

        # Try conditional predictions first
        try:
            fitted = np.asarray(fit.fittedvalues)
            residuals = np.asarray(fit.resid)
            prediction_type = "Conditional"

        except (ValueError, np.linalg.LinAlgError):

            # Fixed-effect predictions avoid singular
            # random-effects covariance errors
            fitted = np.asarray(
                fit.model.exog @ fit.fe_params
            )

            residuals = (
                np.asarray(fit.model.endog) - fitted
            )

            prediction_type = "Fixed effects only"

    else:
        fitted = np.asarray(fit.fittedvalues)
        residuals = np.asarray(fit.resid)
        prediction_type = "OLS"

    return fitted, residuals, prediction_type


# ---------------------------------------------------------
# Diagnostic plots: OLS vs Mixed Effects
# ---------------------------------------------------------

for severity, models in fitted_models.items():

    model_df = models["data"]

    # -----------------------------------------------------
    # 1. OLS DiD diagnostics
    # -----------------------------------------------------

    ols_fit = models["OLS"]

    fitted = np.asarray(ols_fit.fittedvalues)
    residuals = np.asarray(ols_fit.resid)

    fig, axs = plt.subplots(2, 2, figsize=(12, 8))

    sns.scatterplot(
        x=fitted, y=residuals, ax=axs[0, 0]
    )
    axs[0, 0].axhline(0, color="black", linestyle="--")
    axs[0, 0].set_title("Residuals vs Fitted")

    stats.probplot(
        residuals, dist="norm", plot=axs[0, 1]
    )
    axs[0, 1].set_title("Q-Q Plot")

    sns.histplot(
        residuals, kde=True, ax=axs[1, 0]
    )
    axs[1, 0].set_title("Residual Distribution")

    sns.scatterplot(
        x=fitted, y=np.abs(residuals), ax=axs[1, 1]
    )
    axs[1, 1].set_title("Absolute Residuals vs Fitted")

    fig.suptitle(
        f"{severity} — OLS DiD Diagnostics",
        fontsize=15
    )

    plt.tight_layout()
    plt.show()
    plt.close(fig)

    # -----------------------------------------------------
    # 2. Mixed-effects diagnostics
    # -----------------------------------------------------

    mm_fit = models["Mixed"]

    if mm_fit is not None:

        print(f"\n{severity} — Mixed Effects Diagnostics")

        diagnostic_df = plot_mixed_diagnostics(
            mm_fit,
            model_df,
            "Crash_Rate_per_100k"
        )
model_comparison = []

for severity, models in fitted_models.items():

    for model_name in ["OLS", "Mixed"]:

        fit = models[model_name]

        if fit is None:
            continue

        model_comparison.append({
            "Severity": severity,
            "Model": model_name,
            "AIC": fit.aic,
            "BIC": fit.bic,
            "Log_Likelihood": fit.llf,
            "N": fit.nobs
        })

model_comparison_df = pd.DataFrame(model_comparison)

display(model_comparison_df.round(3))

In [ ]:
for severity, models in fitted_models.items():
    mm_fit = models["Mixed"]

    if mm_fit is not None:
        print(f"\n{severity}")
        print("Converged:", mm_fit.converged)
        print("Group variance:", mm_fit.cov_re.iloc[0, 0])
        print("Residual variance:", mm_fit.scale)

In [ ]:
countyweek_cov.columns

In [ ]:
countyweek_cov['log_Population']= np.log1p(countyweek_cov['Population'])
countyweek_cov['log_pct_age_18_24']= np.log1p(countyweek_cov['pct_age_18_24'])
countyweek_cov['log_pct_age_65_plus']= np.log1p(countyweek_cov['pct_age_65_plus'])
countyweek_cov['log_precipitation_mm']= np.log1p(countyweek_cov['precipitation_mm'])
countyweek_cov['log_wind_speed_kph']= np.log1p(countyweek_cov['wind_speed_kph'])
countyweek_cov['log_Crash_Rate_per_100k']= np.log1p(countyweek_cov['Crash_Rate_per_100k'])


In [ ]:



# @title Mixed Effects DiD by Crash Severity

import pandas as pd
import numpy as np
import statsmodels.formula.api as smf

from scipy.stats import chi2
from statsmodels.stats.multitest import multipletests

# ---------------------------------------------------------
# Setup
# ---------------------------------------------------------



data = countyweek_cov.copy()

# Get Group and Storm_Week from alltime
group_lookup = alltime[
    ["County", "Group"]
].drop_duplicates().copy()

storm_lookup = alltime[
    ["County", "Week_Rel", "Storm_Week"]
].drop_duplicates().copy()

# Standardize County names
data["County"] = data["County"].str.strip().str.upper()
group_lookup["County"] = group_lookup["County"].str.strip().str.upper()
storm_lookup["County"] = storm_lookup["County"].str.strip().str.upper()

# Merge Group by County
data = data.drop(columns="Group", errors="ignore").merge(
    group_lookup,
    on="County",
    how="left",
    validate="many_to_one"
)

# Merge Storm_Week by County and Week_Rel
data = data.drop(columns="Storm_Week", errors="ignore").merge(
    storm_lookup,
    on=["County", "Week_Rel"],
    how="left",
    validate="many_to_one"
)

data["Arm"] = pd.Categorical(
    data["Arm"],
    categories=["Control", "Nighttime", "Speeding"]
)

# Reference: weeks -4 to 1
# Analysis: weeks 2 to 5
data = data[data["Week_Rel"].between(-4, 5)].copy()

data["Post"] = (data["Week_Rel"] >= 2).astype(int)

# ---------------------------------------------------------
# Mixed effects DiD for each severity
# ---------------------------------------------------------

results = []
fitted_models = {}

for severity in data["Severity"].dropna().unique():

    print(f"\n{'=' * 60}")
    print(f"Severity: {severity}")
    print(f"{'=' * 60}")

    model_df = data[
        data["Severity"] == severity
    ].copy()

    formula = (
        "Crash_Rate_per_100k"
        " ~ C(Arm)"
        " + C(Post)+ log_Population + C(Storm_Week)+ temp_c + log_pct_age_18_24+ log_pct_age_65_plus+log_precipitation_mm +log_wind_speed_kph"
        " + C(Arm):C(Post)"
    )

    formula0 = (
        "Crash_Rate_per_100k"
        " ~ C(Arm)"
        " + C(Post) + log_Population + C(Storm_Week)+ temp_c + log_pct_age_18_24+ log_pct_age_65_plus+log_precipitation_mm +log_wind_speed_kph"
    )




    # Group random intercept
    mm = smf.mixedlm(
        formula,
        model_df,
        groups=model_df["Group"],
        re_formula="1"
    )

    mm0 = smf.mixedlm(
        formula0,
        model_df,
        groups=model_df["Group"],
        re_formula="1"
    )

    mm_fit = mm.fit(
        reml=False,
        method="lbfgs",
        maxiter=2000
    )

    mm_fit0 = mm0.fit(
        reml=False,
        method="lbfgs",
        maxiter=2000
    )

    # Joint LR test of the two treatment interactions
    lr_stat = 2 * (mm_fit.llf - mm_fit0.llf)

    joint_p = chi2.sf(
        max(0, lr_stat),
        df=2
    )

    fitted_models[severity] = {
        "full": mm_fit,
        "reduced": mm_fit0,
        "data": model_df,
        "joint_p": joint_p
    }

    # -----------------------------------------------------
    # Treatment comparisons
    # -----------------------------------------------------

    severity_results = []

    for treatment in ["Nighttime", "Speeding"]:

        term = (
            f"C(Arm)[T.{treatment}]"
            ":C(Post)[T.1]"
        )

        ci = mm_fit.conf_int().loc[term]

        severity_results.append({
            "Severity": severity,
            "Treatment": treatment,
            "Estimate": mm_fit.params[term],
            "SE": mm_fit.bse[term],
            "CI_Lower": ci.iloc[0],
            "CI_Upper": ci.iloc[1],
            "P_Value": mm_fit.pvalues[term],
            "Joint_LR_P": joint_p
        })

    results.extend(severity_results)

# ---------------------------------------------------------
# Results table
# ---------------------------------------------------------

results_df = pd.DataFrame(results)

results_df["Holm_P"] = np.nan

for severity, idx in results_df.groupby("Severity").groups.items():

    results_df.loc[idx, "Holm_P"] = multipletests(
        results_df.loc[idx, "P_Value"],
        method="holm"
    )[1]

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)

display(results_df.round(4))

In [ ]:
# @title OLS DiD vs Mixed Effects DiD by Severity

import pandas as pd
import numpy as np
import statsmodels.formula.api as smf

from scipy.stats import chi2
from statsmodels.stats.multitest import multipletests

# ---------------------------------------------------------
# Setup
# ---------------------------------------------------------

data = countyweek.copy()

group_lookup = alltime[["County", "Group"]].copy()

group_lookup["County"] = (
    group_lookup["County"].str.strip().str.upper()
)

group_lookup = group_lookup.drop_duplicates()

data["County"] = data["County"].str.strip().str.upper()

data = data.drop(columns="Group", errors="ignore").merge(
    group_lookup,
    on="County",
    how="left",
    validate="many_to_one"
)

data["Arm"] = pd.Categorical(
    data["Arm"],
    categories=["Control", "Nighttime", "Speeding"]
)

# Reference weeks -4 to 1; analysis weeks 2 to 5
data = data[data["Week_Rel"].between(-4, 5)].copy()
data["Post"] = (data["Week_Rel"] >= 2).astype(int)

# ---------------------------------------------------------
# Run both models for each severity
# ---------------------------------------------------------

results = []
fitted_models = {}

for severity in data["Severity"].dropna().unique():

    model_df = data[
        data["Severity"] == severity
    ].dropna(
        subset=["Crash_Rate_per_100k", "Arm", "Post", "Group"]
    ).copy()

    formula = (
        "Crash_Rate_per_100k"
        " ~ C(Arm)"
        " + C(Post)"
        " + C(Arm):C(Post)"
    )

    formula0 = (
        "Crash_Rate_per_100k"
        " ~ C(Arm)"
        " + C(Post)"
    )

    # OLS DiD
    ols_fit = smf.ols(
        formula,
        data=model_df
    ).fit(
        cov_type="cluster",
        cov_kwds={
            "groups": model_df["Group"],
            "use_correction": True,
            "df_correction": True
        },
        use_t=True
    )

    # Mixed-effects DiD
    mm_fit = smf.mixedlm(
        formula,
        model_df,
        groups=model_df["Group"],
        re_formula="1"
    ).fit(reml=False, method="lbfgs", maxiter=2000)

    mm_fit0 = smf.mixedlm(
        formula0,
        model_df,
        groups=model_df["Group"],
        re_formula="1"
    ).fit(reml=False, method="lbfgs", maxiter=2000)

    lr_stat = 2 * (mm_fit.llf - mm_fit0.llf)
    joint_p = chi2.sf(max(0, lr_stat), df=2)

    fitted_models[severity] = {
        "OLS": ols_fit,
        "Mixed": mm_fit,
        "Mixed_Joint_P": joint_p,
        "data": model_df
    }

    # -----------------------------------------------------
    # Extract treatment effects
    # -----------------------------------------------------

    for model_name, fit in [
        ("OLS", ols_fit),
        ("Mixed", mm_fit)
    ]:

        for treatment in ["Nighttime", "Speeding"]:

            term = (
                f"C(Arm)[T.{treatment}]"
                ":C(Post)[T.1]"
            )

            ci = fit.conf_int().loc[term]

            results.append({
                "Severity": severity,
                "Model": model_name,
                "Treatment": treatment,
                "Estimate": fit.params[term],
                "SE": fit.bse[term],
                "CI_Lower": ci.iloc[0],
                "CI_Upper": ci.iloc[1],
                "P_Value": fit.pvalues[term],
                "Joint_LR_P": joint_p if model_name == "Mixed" else np.nan
            })


# ---------------------------------------------------------
# Results table
# ---------------------------------------------------------

results_df = pd.DataFrame(results)

results_df["Holm_P"] = np.nan

for _, idx in results_df.groupby(
    ["Severity", "Model"]
).groups.items():

    results_df.loc[idx, "Holm_P"] = multipletests(
        results_df.loc[idx, "P_Value"],
        method="holm"
    )[1]

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)

display(results_df.round(4))

In [ ]:
import pandas as pd
import numpy as np

# ---------------------------------------------------------
# 1. Prepare crash data
# ---------------------------------------------------------

crash = crashstatistics.copy()

crash["CrashDateTime"] = pd.to_datetime(
    crash["CrashDateTime"], errors="coerce"
)

# Standardize county names: uppercase, remove "County"
crash["County"] = (
    crash["County"]
    .astype(str)
    .str.replace(r"\s+County$", "", regex=True, case=False)
    .str.strip()
    .str.upper()
)

# Tuesday-start weeks
# Tuesday = 1 in pandas (Monday = 0)
crash["Week_Start"] = (
    crash["CrashDateTime"].dt.normalize()
    - pd.to_timedelta(
        (crash["CrashDateTime"].dt.dayofweek - 1) % 7,
        unit="D"
    )
)

# ---------------------------------------------------------
# 2. Create crash outcome indicators
# ---------------------------------------------------------

fatal = crash["CrashSeverity"].eq("Fatal")

speed = (
    crash["SpeedRelated"]
    .astype(str)
    .str.lower()
    .eq("true")
)

light = crash["LightCondition"].astype("string").str.lower()
weather = crash["Weather"].astype("string").str.lower()
crash["Total_Crashes"] = 1
crash["Fatal_All"] = fatal.astype(int)
crash["Fatal_Speed"] = (fatal & speed).astype(int)
crash["Fatal_NonSpeed"] = (fatal & ~speed).astype(int)

crash["Fatal_Nighttime"] = (
    fatal & light.str.contains("dark", na=False)
).astype(int)

crash["Fatal_Daytime"] = (
    fatal & light.eq("daylight").fillna(False)
).astype(int)

crash["Fatal_Rain"] = (
    fatal & weather.str.contains(r"\brain\b", na=False)
).astype(int)

# ---------------------------------------------------------
# 3. Aggregate to county-week level
# ---------------------------------------------------------

metrics = [
    "Total_Crashes",
    "Fatal_All",
    "Fatal_Speed",
    "Fatal_NonSpeed",
    "Fatal_Nighttime",
    "Fatal_Daytime",
    "Fatal_Rain"
]
crash_agg = (
    crash.groupby(["County", "Week_Start"], as_index=False)
    [metrics]
    .sum()
)

# ---------------------------------------------------------
# 4. Prepare countyweek for joining
# ---------------------------------------------------------

countyweek["County"] = (
    countyweek["County"]
    .astype(str)
    .str.replace(r"\s+County$", "", regex=True, case=False)
    .str.strip()
    .str.upper()
)

countyweek["Week_Start"] = pd.to_datetime(
    countyweek["Week_Start"]
).dt.normalize()

# ---------------------------------------------------------
# 5. Left join new crash metrics
# ---------------------------------------------------------

countyweek = countyweek.merge(
    crash_agg,
    on=["County", "Week_Start"],
    how="left",
    validate="many_to_one"
)

countyweek[metrics] = (
    countyweek[metrics]
    .fillna(0)
    .astype(int)
)

# ---------------------------------------------------------
# 6. Calculate additional fatal crash rates
# ---------------------------------------------------------

for metric in metrics:
    countyweek[f"{metric}_Rate"] = (
        countyweek[metric]
        / countyweek["Population"]
        * 100_000
    )

display(countyweek.head())

In [ ]:
# @title Weather and Demographic Covariates

import pandas as pd
import numpy as np
import meteostat as ms
from datacommons_client.client import DataCommonsClient

# Assumes `client` is already initialized with your API key.


# ---------------------------------------------------------
# Get Ohio counties from Data Commons
# ---------------------------------------------------------

response = client.node.fetch_property_values(
    node_dcids=["geoId/39"],
    properties="containedInPlace+{typeOf:County}",
    out=False
).to_dict()

county_nodes = (
    response["data"]["geoId/39"]["arcs"]
    ["containedInPlace+"]["nodes"]
)

county_locations = pd.DataFrame(
    county_nodes
)[["dcid", "name"]]

county_locations["County"] = (
    county_locations["name"]
    .str.replace(" County", "", regex=False)
    .str.strip()
    .str.upper()
)


# ---------------------------------------------------------
# Get county coordinates
# ---------------------------------------------------------

coordinate_response = (
    client.node.fetch_property_values(
        node_dcids=county_locations["dcid"].tolist(),
        properties=["latitude", "longitude"]
    )
    .to_dict()["data"]
)


def get_coordinate(dcid, variable):

    nodes = (
        coordinate_response[dcid]["arcs"]
        [variable]["nodes"]
    )

    return float(nodes[0]["value"])


county_locations["Latitude"] = (
    county_locations["dcid"]
    .apply(lambda x: get_coordinate(x, "latitude"))
)

county_locations["Longitude"] = (
    county_locations["dcid"]
    .apply(lambda x: get_coordinate(x, "longitude"))
)


# ---------------------------------------------------------
# Clean countyweek dates and counties
# ---------------------------------------------------------

countyweek["Week_Start"] = pd.to_datetime(
    countyweek["Week_Start"],
    format="mixed"
).dt.normalize()

countyweek["County"] = (
    countyweek["County"]
    .astype(str)
    .str.replace(
        r"\s+County$",
        "",
        regex=True,
        case=False
    )
    .str.strip()
    .str.upper()
)

start_date = countyweek["Week_Start"].min().date()

end_date = (
    countyweek["Week_Start"].max()
    + pd.Timedelta(days=6)
).date()


# ---------------------------------------------------------
# Download weather data
# ---------------------------------------------------------

weather_data = []
missing_counties = []

for _, county in county_locations.iterrows():

    point = ms.Point(
        float(county["Latitude"]),
        float(county["Longitude"])
    )

    stations = ms.stations.nearby(
        point,
        radius=150_000,
        limit=4
    )

    if stations is None or len(stations) == 0:

        missing_counties.append(county["County"])

        print(f"No stations found for {county['County']}")

        continue

    daily = ms.daily(
        stations,
        start_date,
        end_date,
        parameters=[
            ms.Parameter.TEMP,
            ms.Parameter.TMIN,
            ms.Parameter.TMAX,
            ms.Parameter.PRCP,
            ms.Parameter.SNWD,
            ms.Parameter.WSPD,
            ms.Parameter.WPGT
        ]
    )

    daily = ms.interpolate(
        daily,
        point
    ).fetch()

    if daily.empty:

        print(f"No weather data found for {county['County']}")
        continue

    daily.index.name = "Date"
    daily = daily.reset_index()

    daily.columns = [
        getattr(column, "value", column)
        for column in daily.columns
    ]

    daily["County"] = county["County"]

    weather_data.append(daily)


print("Counties without stations:", missing_counties)

if len(weather_data) == 0:
    raise ValueError("No weather data were downloaded.")


# ---------------------------------------------------------
# Combine daily weather
# ---------------------------------------------------------

weather_daily = pd.concat(
    weather_data,
    ignore_index=True
)

weather_daily["Date"] = pd.to_datetime(
    weather_daily["Date"]
)


# ---------------------------------------------------------
# Assign Tuesday-start weeks
# ---------------------------------------------------------

weather_daily["Week_Start"] = (
    weather_daily["Date"].dt.normalize()
    - pd.to_timedelta(
        (weather_daily["Date"].dt.dayofweek - 1) % 7,
        unit="D"
    )
)


# ---------------------------------------------------------
# Aggregate daily weather to weekly weather
# ---------------------------------------------------------

weather_weekly = (
    weather_daily
    .groupby(
        ["County", "Week_Start"],
        as_index=False
    )
    .agg(
        temp_c=("temp", "mean"),
        precipitation_mm=("prcp", "sum"),
        snow_depth_cm=("snwd", "max"),
        wind_speed_kph=("wspd", "mean")
    )
)


# ---------------------------------------------------------
# Get 2024 demographic data from Data Commons
# ---------------------------------------------------------

demographic_variables = {
    "Median_Income_Household":
        "median_household_income",

    "UnemploymentRate_Person":
        "unemployment_rate",

    "Count_Person":
        "population",

    "Count_Person_18To24Years":
        "population_18_24",

    "Count_Person_65OrMoreYears":
        "population_65_plus",

    "Count_Person_25OrMoreYears":
        "population_25_plus",

    "Count_Person_25OrMoreYears_EducationalAttainmentBachelorsDegreeOrHigher":
        "population_bachelors_plus"
}


demographics_long = client.observations_dataframe(
    variable_dcids=list(demographic_variables.keys()),
    date="2024",
    entity_dcids=county_locations["dcid"].tolist(),
    property_filters={
        "measurementMethod": "CensusACS5yrSurvey"
    }
)


demographics = (
    demographics_long
    .pivot_table(
        index="entity",
        columns="variable",
        values="value",
        aggfunc="first"
    )
    .reindex(columns=demographic_variables.keys())
    .rename(columns=demographic_variables)
    .reset_index()
    .rename(columns={"entity": "dcid"})
)


demographics = county_locations[
    ["dcid", "County"]
].merge(
    demographics,
    on="dcid",
    how="left",
    validate="one_to_one"
)


# ---------------------------------------------------------
# Convert demographic data to numeric
# ---------------------------------------------------------

numeric_demo_cols = list(demographic_variables.values())

demographics[numeric_demo_cols] = (
    demographics[numeric_demo_cols]
    .apply(pd.to_numeric, errors="coerce")
)


# ---------------------------------------------------------
# Create model-ready demographic variables
# ---------------------------------------------------------

demographics["log_median_household_income"] = np.log(
    demographics["median_household_income"].where(
        demographics["median_household_income"] > 0
    )
)

demographics["pct_age_18_24"] = (
    100
    * demographics["population_18_24"]
    / demographics["population"]
)

demographics["pct_age_65_plus"] = (
    100
    * demographics["population_65_plus"]
    / demographics["population"]
)

demographics["pct_bachelors_plus"] = (
    100
    * demographics["population_bachelors_plus"]
    / demographics["population_25_plus"]
)


demographic_columns = [
    "County",
    "log_median_household_income",
    "unemployment_rate",
    "pct_age_18_24",
    "pct_age_65_plus",
    "pct_bachelors_plus"
]


# ---------------------------------------------------------
# Merge weather and demographics into countyweek
# ---------------------------------------------------------

weather_cols = [
    "temp_c",
    "precipitation_mm",
    "snow_depth_cm",
    "wind_speed_kph"
]

# Avoid duplicate columns if rerunning this cell
countyweek_cov = countyweek.drop(
    columns=weather_cols + demographic_columns[1:],
    errors="ignore"
).merge(
    weather_weekly[
        ["County", "Week_Start"] + weather_cols
    ],
    on=["County", "Week_Start"],
    how="left",
    validate="many_to_one"
)

countyweek_cov = countyweek_cov.merge(
    demographics[demographic_columns],
    on="County",
    how="left",
    validate="many_to_one"
)


# ---------------------------------------------------------
# Validate merges
# ---------------------------------------------------------

print("Original rows:", len(countyweek))
print("Merged rows:", len(countyweek_cov))

print("\nMissing weather values:")
display(countyweek_cov[weather_cols].isna().sum())

print("\nMissing demographic values:")
display(countyweek_cov[demographic_columns[1:]].isna().sum())

display(
    countyweek_cov[
        [
            "County",
            "Week_Start",
            "temp_c",
            "precipitation_mm",
            "snow_depth_cm",
            "wind_speed_kph",
            "log_median_household_income",
            "unemployment_rate",
            "pct_age_18_24",
            "pct_age_65_plus",
            "pct_bachelors_plus"
        ]
    ].head(15)
)

In [ ]:
countyweek_cov.isna().sum()



In [ ]:
df = countyweek_cov[countyweek_cov["Severity"] == "All"].copy()

for metric in df.columns:

    if (
        isinstance(df[metric].dtype, pd.CategoricalDtype)
        or not pd.api.types.is_numeric_dtype(df[metric])
    ):

        print(metric, "metric cat")
        display(df[metric].value_counts(dropna=False).to_frame("Count"))

    else:

        print(metric, "metric cont")

        log_metric = f"log_{metric}"

        if log_metric not in df.columns:
            df[log_metric] = np.log1p(df[metric])

        with sns.axes_style("white"):

            fig, axs = plt.subplots(ncols=2, figsize=(15, 5))

            sns.histplot(df[metric].dropna(), kde=True, ax=axs[0])
            sns.histplot(df[log_metric].dropna(), kde=True, ax=axs[1])

            axs[0].set_title(f"Original: {metric}")
            axs[1].set_title(f"Log: {metric}")

            plt.tight_layout()
            plt.show()
            plt.close(fig)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

campaign_order = [
    "Control",
    "Speeding",
    "Nighttime"
]

campaign_colors = {
    "Control": "#7F7F7F",
    "Speeding": "#E76F51",
    "Nighttime": "#457B9D"
}

statistics = [
    ("Average", "Average across counties"),
    ("Median", "Median across counties"),
    ("Minimum", "Minimum across counties"),
    ("Maximum", "Maximum across counties")
]

# ---------------------------------------------------------
# Prepare dataset
# ---------------------------------------------------------

df = countyweek_cov[
    countyweek_cov["Severity"] == "All"
].copy()

df["Week_Start"] = pd.to_datetime(df["Week_Start"])

# Select numeric metrics
excluded_columns = [
    "Week_Rel",
    "Population"
]

metrics = [
    col for col in df.select_dtypes(include="number").columns
    if col not in excluded_columns
    and not col.startswith("log_")
]


# ---------------------------------------------------------
# Create weekly time-series statistics
# ---------------------------------------------------------

def create_time_series_metrics(df):

    time_series = df.melt(
        id_vars=[
            "County",
            "Week_Start",
            "Arm"
        ],
        value_vars=metrics,
        var_name="Metric",
        value_name="Value"
    )

    time_series = (
        time_series
        .groupby(
            ["Week_Start", "Metric", "Arm"],
            as_index=False
        )
        .agg(
            Average=("Value", "mean"),
            Median=("Value", "median"),
            Minimum=("Value", "min"),
            Maximum=("Value", "max"),
            Standard_Deviation=("Value", "std"),
            County_Count=("County", "nunique")
        )
    )

    return time_series


countyweek_time_series = create_time_series_metrics(df)


# ---------------------------------------------------------
# Plot time series
# ---------------------------------------------------------

def plot_time_series(time_series_df, dataset_name):

    metrics = time_series_df["Metric"].unique()

    post_start = pd.Timestamp("2026-02-17")

    date_min = time_series_df["Week_Start"].min()
    date_max = time_series_df["Week_Start"].max()

    sns.set_theme(style="whitegrid")

    for metric in metrics:

        metric_data = time_series_df[
            time_series_df["Metric"] == metric
        ]

        fig, axs = plt.subplots(
            nrows=2,
            ncols=2,
            figsize=(18, 11),
            sharex=True
        )

        axs = axs.flatten()

        for ax, (statistic, statistic_label) in zip(
            axs, statistics
        ):

            ax.axvspan(
                date_min, post_start,
                color="#D9D9D9", alpha=0.18
            )

            ax.axvspan(
                post_start, date_max,
                color="#F6E8A6", alpha=0.18
            )

            for campaign_group in campaign_order:

                group_data = (
                    metric_data[
                        metric_data["Arm"] == campaign_group
                    ]
                    .sort_values("Week_Start")
                )

                ax.plot(
                    group_data["Week_Start"],
                    group_data[statistic],
                    color=campaign_colors[campaign_group],
                    linewidth=2,
                    marker="o",
                    markersize=3,
                    label=campaign_group
                )

            ax.axvline(
                post_start,
                color="black",
                linestyle="--",
                linewidth=1.5
            )

            ax.set_title(statistic_label)
            ax.set_xlabel("Week")
            ax.set_ylabel(statistic)

            ax.xaxis.set_major_formatter(
                mdates.DateFormatter("%b %d")
            )

            ax.grid(alpha=0.25)

        handles, labels = axs[0].get_legend_handles_labels()

        fig.legend(
            handles,
            labels,
            loc="upper center",
            bbox_to_anchor=(0.5, 0.95),
            ncol=3,
            frameon=False
        )

        fig.suptitle(
            f"{dataset_name}: {metric} Over Time",
            fontsize=16,
            y=1.00
        )

        plt.tight_layout(rect=[0, 0, 1, 0.91])
        plt.show()
        plt.close(fig)


plot_time_series(
    countyweek_time_series,
    "Crash Data"
)

In [ ]:
# @title Parallel Trends Analysis - Crash Data

import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

df = countyweek_cov[
    countyweek_cov["Severity"] == "All"
].copy()

metrics = [
    "Crash_Rate_per_100k",
    "Total_Crashes",
    "Fatal_All",
    "Fatal_Speed",
    "Fatal_NonSpeed",
    "Fatal_Nighttime",
    "Fatal_Daytime",
    "Fatal_Rain",
    "Total_Crashes_Rate",
    "Fatal_All_Rate",
    "Fatal_Speed_Rate",
    "Fatal_NonSpeed_Rate",
    "Fatal_Nighttime_Rate",
    "Fatal_Daytime_Rate",
    "Fatal_Rain_Rate"
]

df["Week_Start"] = pd.to_datetime(
    df["Week_Start"],
    format="mixed"
)

df["Week_Rel"] = pd.to_numeric(
    df["Week_Rel"],
    errors="coerce"
)

# Use existing weekly index as trend
df["trend"] = df["Week_Rel"]

# Create log versions
for metric in metrics:

    df[metric] = pd.to_numeric(
        df[metric],
        errors="coerce"
    )

    df[f"log_{metric}"] = np.log1p(
        df[metric]
    )

log_metrics = [f"log_{metric}" for metric in metrics]

metrics = metrics + log_metrics

# ---------------------------------------------------------
# Parallel trends models
# ---------------------------------------------------------

for metric in metrics:

    print("\n", "=" * 60)
    print(metric, "metric")
    print("=" * 60)

    model_df = df.loc[
        df["Week_Rel"] < 0
    ].dropna(
        subset=[
            metric,
            "Arm",
            "trend",
            "County"
        ]
    ).copy()

    formula = (
        f"{metric}"
        + " ~ C(Arm, Treatment(reference='Control'))"
        + " + trend"
        + " + C(Arm, Treatment(reference='Control')):trend"
    )

    trend_fit = smf.ols(
        formula,
        data=model_df
    ).fit(
        cov_type="cluster",
        cov_kwds={
            "groups": model_df["County"],
            "use_correction": True,
            "df_correction": True
        },
        use_t=True
    )

    print(trend_fit.summary())

#fatal all and fatal speed, log_Fatal_Speed

In [ ]:
#Fatal_All_Rate	Primary outcome; replicates the paper's fatal crash metric
#Fatal_Speed_Rate	Tests whether the speeding campaign affects speed-related fatal crashes
#Fatal_Nighttime_Rate	Tests whether the nighttime campaign affects nighttime fatal crashes
#Fatal_NonSpeed_Rate	Comparison outcome for speed-related fatalities
#Fatal_Daytime_Rate	Comparison outcome for nighttime fatalities
#Total_Crashes_Rate

In [ ]:
import pandas as pd
from google.colab import drive
import openpyxl

# 1. Mount Drive
drive.mount('/content/drive')




file_path = '/content/drive/MyDrive/traffic/alltime_combined_traffic_data.csv'

# 3. Read the CSV file
alltime = pd.read_csv(file_path)

In [ ]:
storm_week = (
    alltime[["County", "Week_Rel", "Storm_Week", "Group"]]
    .drop_duplicates()
    .copy()
)

storm_week["County"] = storm_week["County"].str.upper().str.strip()

data = data.merge(
    storm_week,
    on=["County", "Week_Rel"],
    how="left",
    validate="many_to_one"
)

display(data.head())
import pandas as pd
from google.colab import drive
import openpyxl

# 1. Mount Drive
drive.mount('/content/drive')




file_path = '/content/drive/MyDrive/traffic/crash/crashrate.csv'
data.to_csv('/content/drive/MyDrive/traffic/crash/data_1.csv', index=False)

In [ ]:

# 1. Mount Drive
drive.mount('/content/drive')




file_path = '/content/drive/MyDrive/traffic/crash/crashrate.csv'
data.to_csv('/content/drive/MyDrive/traffic/crash/data_1.csv', index=False)

In [ ]:
storm_week = (
    alltime[["County", "Week_Rel", "Storm_Week", "Group"]]
    .drop_duplicates()
    .copy()
)

storm_week["County"] = storm_week["County"].str.upper().str.strip()

df = df.merge(
    storm_week,
    on=["County", "Week_Rel"],
    how="left",
    validate="many_to_one"
)

display(df.head())

In [ ]:
df["Week_Rel"].value_counts()

In [ ]:
# @title OLS DiD vs Mixed Effects DiD

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.formula.api as smf

from scipy import stats
from statsmodels.stats.multitest import multipletests

# ---------------------------------------------------------
# Setup
# ---------------------------------------------------------

data = df[df["Severity"] == "All"].copy()
data = data[data["Week_Rel"].between(-4, 5)].copy()

data["Post"] = (data["Week_Rel"] >= 2).astype(int)

# Match original campaign group names
data["Campaign_Group"] = data["Arm"].map({
    "Control": "Control",
    "Nighttime": "Treated (Nighttime)",
    "Speeding": "Treated (Speeding)"
})

data["Campaign_Group"] = pd.Categorical(
    data["Campaign_Group"],
    categories=[
        "Control",
        "Treated (Nighttime)",
        "Treated (Speeding)"
    ]
)

# ---------------------------------------------------------
# Metrics and log transforms
# ---------------------------------------------------------

metrics = [
    "Crash_Rate_per_100k",
    "Fatal_All_Rate",
    "Fatal_Speed_Rate",
    "Fatal_Nighttime_Rate",
    "Fatal_NonSpeed_Rate",
    "Fatal_Daytime_Rate",
    "Total_Crashes_Rate"
]



for metric in metrics:
    data[f"log_{metric}"] = np.log1p(data[metric])

metrics += [f"log_{metric}" for metric in metrics]

data["log_Population"] = np.log(data["Population"])


# ---------------------------------------------------------
# Fit both models
# ---------------------------------------------------------

results = []
fitted_models = {}

for metric in metrics:
    print('metric', metric)
    formula = (
        metric
        + " ~ C(Campaign_Group)"
        + " + C(Post)"
        + " + log_Population"
        + " + C(Storm_Week)"
        + " + Week_Rel"
        + " + C(Campaign_Group):C(Post)"
    )

    model_df = data.copy()

    # OLS DiD
    ols_fit = smf.ols(
        formula,
        data=model_df
    ).fit()

    # Mixed-effects DiD: Group random intercept
    mixed_fit = smf.mixedlm(
        formula,
        data=model_df,
        groups=model_df["Group"],
        re_formula="1"
    ).fit(
        reml=False,
        method="lbfgs",
        maxiter=2000
    )

    fitted_models[metric] = {
        "OLS": ols_fit,
        "Mixed": mixed_fit
    }

    for model_name, fit in [
        ("OLS", ols_fit),
        ("Mixed", mixed_fit)
    ]:

        for treatment in [
            "Treated (Nighttime)",
            "Treated (Speeding)"
        ]:

            term = (
                f"C(Campaign_Group)[T.{treatment}]"
                ":C(Post)[T.1]"
            )

            ci = fit.conf_int().loc[term]

            results.append({
                "Metric": metric,
                "Model": model_name,
                "Treatment": treatment,
                "Estimate": fit.params[term],
                "SE": fit.bse[term],
                "CI_Lower": ci.iloc[0],
                "CI_Upper": ci.iloc[1],
                "P_Value": fit.pvalues[term]
            })

# ---------------------------------------------------------
# Results
# ---------------------------------------------------------

results_df = pd.DataFrame(results)

results_df["Holm_P"] = np.nan

for _, idx in results_df.groupby(
    ["Metric", "Model"]
).groups.items():

    results_df.loc[idx, "Holm_P"] = multipletests(
        results_df.loc[idx, "P_Value"],
        method="holm"
    )[1]

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)

display(results_df.round(4))


# ---------------------------------------------------------
# Diagnostic plots
# ---------------------------------------------------------

for metric, models in fitted_models.items():

    for model_name, fit in models.items():

        residuals = np.asarray(fit.resid)
        fitted = np.asarray(fit.fittedvalues)

        fig, axs = plt.subplots(2, 2, figsize=(12, 8))

        sns.scatterplot(
            x=fitted, y=residuals, ax=axs[0, 0]
        )
        axs[0, 0].axhline(0, color="black", linestyle="--")
        axs[0, 0].set_title("Residuals vs Fitted")

        stats.probplot(
            residuals, dist="norm", plot=axs[0, 1]
        )
        axs[0, 1].set_title("Q-Q Plot")

        sns.histplot(
            residuals, kde=True, ax=axs[1, 0]
        )
        axs[1, 0].set_title("Residual Distribution")

        sns.scatterplot(
            x=fitted, y=np.abs(residuals), ax=axs[1, 1]
        )
        axs[1, 1].set_title("Absolute Residuals vs Fitted")

        fig.suptitle(f"{metric} — {model_name}", fontsize=15)

        plt.tight_layout()
        plt.show()
        plt.close(fig)

In [ ]:
# @title OLS DiD vs Mixed Effects DiD — Severity and Period Sensitivity

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.formula.api as smf

from scipy import stats
from statsmodels.stats.multitest import multipletests

# ---------------------------------------------------------
# Setup
# ---------------------------------------------------------

data = df.copy()

data["County"] = data["County"].str.upper().str.strip()

# Get Group from alltime if needed
if "Group" not in data.columns:

    group_lookup = alltime[["County", "Group"]].copy()
    group_lookup["County"] = (
        group_lookup["County"].str.upper().str.strip()
    )

    group_lookup = group_lookup.drop_duplicates()

    data = data.merge(
        group_lookup,
        on="County",
        how="left",
        validate="many_to_one"
    )

data["Arm"] = data["Arm"].astype(str).str.strip()

data["Campaign_Group"] = data["Arm"].map({
    "Control": "Control",
    "Nighttime": "Treated (Nighttime)",
    "Speeding": "Treated (Speeding)"
})

data["Campaign_Group"] = pd.Categorical(
    data["Campaign_Group"],
    categories=[
        "Control",
        "Treated (Nighttime)",
        "Treated (Speeding)"
    ]
)

data["log_Population"] = np.log(data["Population"])

# ---------------------------------------------------------
# Pre/Post specifications
# ---------------------------------------------------------

periods = {
    "Original": ([-4, -3, -2, -1, 0, 1], [2, 3, 4, 5]),
    "No Ramp-Up": ([-4, -3, -2, -1], [2, 3, 4, 5]),
    "Exclude First Week": ([-3, -2, -1, 0, 1], [2, 3, 4, 5]),
    "Extended Post": ([-4, -3, -2, -1, 0, 1], [2, 3, 4, 5, 6])
}

# ---------------------------------------------------------
# Formula — same fixed effects for both models
# ---------------------------------------------------------

metric = "Crash_Rate_per_100k"

formula = (
    metric
    + " ~ C(Campaign_Group)"
    + " + C(Post)"
    + " + log_Population"
    + " + C(Storm_Week)"
    + " + Week_Rel"
    + " + C(Campaign_Group):C(Post)"
)

# ---------------------------------------------------------
# Fit models
# ---------------------------------------------------------

results = []
fitted_models = {}

required = [
    metric,
    "Campaign_Group",
    "Post",
    "Group",
    "Storm_Week",
    "log_Population",
    "Week_Rel"
]

for severity in data["Severity"].dropna().unique():

    for period_name, (pre_weeks, post_weeks) in periods.items():

        print(f"\n{severity} — {period_name}")

        model_df = data[
            (data["Severity"] == severity)
            & data["Week_Rel"].isin(pre_weeks + post_weeks)
        ].copy()

        model_df["Post"] = (
            model_df["Week_Rel"].isin(post_weeks)
        ).astype(int)

        model_df = model_df.dropna(
            subset=required
        ).copy()

        if model_df.empty:
            print("No observations. Skipping.")
            continue

        # -------------------------------------------------
        # OLS DiD — Group-clustered standard errors
        # -------------------------------------------------

        ols_fit = smf.ols(
            formula,
            data=model_df
        ).fit(
            cov_type="cluster",
            cov_kwds={
                "groups": model_df["Group"],
                "use_correction": True,
                "df_correction": True
            },
            use_t=True
        )

        # -------------------------------------------------
        # Mixed-effects DiD — Group random intercept
        # -------------------------------------------------

        mixed_fit = None

        try:
            mixed_fit = smf.mixedlm(
                formula,
                data=model_df,
                groups=model_df["Group"],
                re_formula="1"
            ).fit(
                reml=False,
                method="lbfgs",
                maxiter=2000
            )

            if not mixed_fit.converged:
                print("Mixed model did not converge.")
                mixed_fit = None

        except (ValueError, np.linalg.LinAlgError) as e:
            print(f"Mixed model failed: {e}")

        # -------------------------------------------------
        # Save models
        # -------------------------------------------------

        fitted_models[(severity, period_name)] = {
            "OLS": ols_fit,
            "Mixed": mixed_fit,
            "data": model_df
        }

        # -------------------------------------------------
        # Extract treatment effects
        # -------------------------------------------------

        for model_name, fit in [
            ("OLS", ols_fit),
            ("Mixed", mixed_fit)
        ]:

            if fit is None:
                continue

            for treatment in [
                "Treated (Nighttime)",
                "Treated (Speeding)"
            ]:

                term = (
                    f"C(Campaign_Group)[T.{treatment}]"
                    ":C(Post)[T.1]"
                )

                ci = fit.conf_int().loc[term]

                results.append({
                    "Severity": severity,
                    "Period": period_name,
                    "Model": model_name,
                    "Treatment": treatment,
                    "Estimate": fit.params[term],
                    "SE": fit.bse[term],
                    "CI_Lower": ci.iloc[0],
                    "CI_Upper": ci.iloc[1],
                    "P_Value": fit.pvalues[term],
                    "N": len(model_df)
                })


# ---------------------------------------------------------
# Results table
# ---------------------------------------------------------

results_df = pd.DataFrame(results)

results_df["Holm_P"] = np.nan

for _, idx in results_df.groupby(
    ["Severity", "Period", "Model"]
).groups.items():

    p_values = results_df.loc[idx, "P_Value"]

    if p_values.notna().all():
        results_df.loc[idx, "Holm_P"] = multipletests(
            p_values,
            method="holm"
        )[1]

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)

display(results_df.round(4))

In [ ]:
# @title Leave-One-Group-Out and Leave-One-County-Out Sensitivity

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.formula.api as smf
import warnings

from statsmodels.tools.sm_exceptions import ConvergenceWarning

# ---------------------------------------------------------
# Setup
# ---------------------------------------------------------

metric = "Crash_Rate_per_100k"

formula = (
    metric
    + " ~ C(Campaign_Group)"
    + " + C(Post)"
    + " + log_Population"
    + " + C(Storm_Week)"
    + " + Week_Rel"
    + " + C(Campaign_Group):C(Post)"
)

base_data = data[
    data["Week_Rel"].between(-4, 5)
].copy()

base_data["Post"] = (
    base_data["Week_Rel"] >= 2
).astype(int)

# Ensure Group and campaign assignments are available
if "Group" not in base_data.columns:
    lookup = alltime[["County", "Group"]].copy()
    lookup["County"] = lookup["County"].str.strip().str.upper()
    lookup = lookup.drop_duplicates("County")

    base_data["County"] = base_data["County"].str.strip().str.upper()
    base_data = base_data.merge(
        lookup, on="County", how="left", validate="many_to_one"
    )

if "Campaign_Group" not in base_data.columns:
    base_data["Campaign_Group"] = base_data["Arm"].map({
        "Control": "Control",
        "Nighttime": "Treated (Nighttime)",
        "Speeding": "Treated (Speeding)"
    })

base_data["Campaign_Group"] = pd.Categorical(
    base_data["Campaign_Group"],
    categories=[
        "Control",
        "Treated (Nighttime)",
        "Treated (Speeding)"
    ]
)

if "log_Population" not in base_data.columns:
    base_data["log_Population"] = np.log(base_data["Population"])

base_data = base_data.dropna(
    subset=[
        metric, "Severity", "County", "Group",
        "Campaign_Group", "Post", "Storm_Week",
        "log_Population", "Week_Rel"
    ]
).copy()

treatments = {
    "Nighttime":
        "C(Campaign_Group)[T.Treated (Nighttime)]:C(Post)[T.1]",
    "Speeding":
        "C(Campaign_Group)[T.Treated (Speeding)]:C(Post)[T.1]"
}

# ---------------------------------------------------------
# Fit OLS and Mixed Effects
# ---------------------------------------------------------

def fit_sensitivity_models(model_df):

    output = []

    if model_df["Campaign_Group"].nunique() < 3:
        return output

    # OLS DiD with Group-clustered standard errors
    try:
        ols_fit = smf.ols(
            formula,
            data=model_df
        ).fit(
            cov_type="cluster",
            cov_kwds={
                "groups": model_df["Group"],
                "use_correction": True,
                "df_correction": True
            },
            use_t=True
        )

        for treatment, term in treatments.items():
            output.append({
                "Model": "OLS",
                "Treatment": treatment,
                "Estimate": ols_fit.params[term],
                "P_Value": ols_fit.pvalues[term]
            })

    except Exception as e:
        print("OLS failed:", e)

    # Mixed-effects DiD with Group random intercept
    try:
        with warnings.catch_warnings():
            warnings.simplefilter("error", ConvergenceWarning)

            mixed_fit = smf.mixedlm(
                formula,
                model_df,
                groups=model_df["Group"],
                re_formula="1"
            ).fit(
                reml=False,
                method="lbfgs",
                maxiter=2000,
                disp=False
            )

        if mixed_fit.converged:
            for treatment, term in treatments.items():
                output.append({
                    "Model": "Mixed",
                    "Treatment": treatment,
                    "Estimate": mixed_fit.params[term],
                    "P_Value": mixed_fit.pvalues[term]
                })

    except Exception:
        pass

    return output


# ---------------------------------------------------------
# Full sample and leave-one-out models
# ---------------------------------------------------------

results = []

for severity in base_data["Severity"].dropna().unique():

    severity_df = base_data[
        base_data["Severity"] == severity
    ].copy()

    # Full sample
    for result in fit_sensitivity_models(severity_df):
        results.append({
            "Severity": severity,
            "Omitted_Level": "None",
            "Omitted_Unit": "Full Sample",
            **result
        })

    # Leave one Group out
    for group in severity_df["Group"].unique():

        subset = severity_df[
            severity_df["Group"] != group
        ].copy()

        for result in fit_sensitivity_models(subset):
            results.append({
                "Severity": severity,
                "Omitted_Level": "Group",
                "Omitted_Unit": str(group),
                **result
            })

    # Leave one County out
    for county in severity_df["County"].unique():

        subset = severity_df[
            severity_df["County"] != county
        ].copy()

        for result in fit_sensitivity_models(subset):
            results.append({
                "Severity": severity,
                "Omitted_Level": "County",
                "Omitted_Unit": str(county),
                **result
            })


sensitivity_df = pd.DataFrame(results)

# ---------------------------------------------------------
# Influence calculations
# ---------------------------------------------------------

full_estimates = (
    sensitivity_df[
        sensitivity_df["Omitted_Level"] == "None"
    ][
        ["Severity", "Model", "Treatment", "Estimate"]
    ]
    .rename(columns={"Estimate": "Full_Estimate"})
)

influence_df = (
    sensitivity_df[
        sensitivity_df["Omitted_Level"] != "None"
    ]
    .merge(
        full_estimates,
        on=["Severity", "Model", "Treatment"],
        how="left"
    )
)

influence_df["Change"] = (
    influence_df["Estimate"]
    - influence_df["Full_Estimate"]
)

influence_df["Absolute_Change"] = (
    influence_df["Change"].abs()
)

# ---------------------------------------------------------
# Results tables
# ---------------------------------------------------------

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)

print("\nFULL-SAMPLE ESTIMATES")
display(
    sensitivity_df[
        sensitivity_df["Omitted_Level"] == "None"
    ].round(4)
)

print("\nMOST INFLUENTIAL GROUPS AND COUNTIES")
display(
    influence_df
    .sort_values("Absolute_Change", ascending=False)
    .groupby(
        ["Severity", "Model", "Treatment", "Omitted_Level"]
    )
    .head(5)
    .round(4)
)

# ---------------------------------------------------------
# Sensitivity plots
# ---------------------------------------------------------

sns.set_theme(style="whitegrid")

for severity in sensitivity_df["Severity"].unique():

    for model in ["OLS", "Mixed"]:

        for treatment in ["Nighttime", "Speeding"]:

            subset = sensitivity_df[
                (sensitivity_df["Severity"] == severity)
                & (sensitivity_df["Model"] == model)
                & (sensitivity_df["Treatment"] == treatment)
            ].copy()

            full = subset.loc[
                subset["Omitted_Level"] == "None",
                "Estimate"
            ]

            if full.empty:
                continue

            full_estimate = full.iloc[0]

            fig, axs = plt.subplots(
                1, 2, figsize=(17, 7)
            )

            for ax, level in zip(
                axs, ["Group", "County"]
            ):

                plot_df = subset[
                    subset["Omitted_Level"] == level
                ].sort_values("Estimate")

                if plot_df.empty:
                    continue

                sns.scatterplot(
                    data=plot_df,
                    x="Estimate",
                    y="Omitted_Unit",
                    ax=ax
                )

                ax.axvline(
                    full_estimate,
                    color="red",
                    linestyle="--",
                    label="Full sample"
                )

                ax.axvline(
                    0,
                    color="black",
                    linestyle=":"
                )

                ax.set_title(
                    f"Leave-One-{level}-Out"
                )
                ax.set_xlabel(
                    "Estimated DiD effect per 100,000"
                )
                ax.set_ylabel(f"Omitted {level}")
                ax.legend()

            fig.suptitle(
                f"{severity} — {model} — {treatment}",
                fontsize=15
            )

            plt.tight_layout()
            plt.show()
            plt.close(fig)